<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9_multiclass_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9 (multiclass 3D) — marginalizing one nuisance without a scalar flow

This notebook is a dimension-raised companion to `Exercise_9_multiclass.ipynb`; that notebook and the original Exercise 9 remain unchanged.  The algorithm, flow architecture, **training** simulation budgets, ten-member classifier ensemble, progressive classifier learning-rate schedule, pure-CE objective, and standalone figure export are kept fixed.  Diagnostic sample budgets are increased below to separate learned structure from Monte Carlo noise.

Only the benchmark is changed.  The observable is four-dimensional and the parameter is

$$
\theta=(\mu,\alpha,\beta),
$$

with one POI $\mu$ and two nuisance parameters $\alpha,\beta$.  The simulator is a two-component conditional Gaussian mixture.  Part I integrates $\beta$ through simulation and trains the genuinely two-dimensional proposals $q_P^m(\mu,\alpha\mid x)$ and $q_L^m(x\mid\mu,\alpha)$.  Both variables therefore participate in the alternating coupling masks.  Part II keeps all three parameters explicit and trains $q_P(\mu,\alpha,\beta\mid x)$ and $q_L(x\mid\mu,\alpha,\beta)$.

The classifier is not widened: its inputs are only six-dimensional in Part I and seven-dimensional in Part II, while the unchanged 1024-wide network already has about 3.2 million parameters per ensemble member.  Analytic mixture densities are used only for validation plots.  This revision changes only diagnostic Monte Carlo budgets: it requires and loads the existing fingerprinted network checkpoints and refuses to retrain a missing or incompatible model.


In [1]:
# ========================================================================
# Google Colab setup — safe to re-run; a no-op outside Colab.
# ========================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", "nflows==0.14")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    candidates = [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]
    TUTORIAL_DIR = next(
        (candidate for candidate in candidates if (candidate / "utils_hnpe.py").exists()),
        None,
    )
    if TUTORIAL_DIR is None:
        raise FileNotFoundError(
            "Run from the repository root or workshops/ml4hep_tifr_colab."
        )
    if str(TUTORIAL_DIR.resolve()) not in sys.path:
        sys.path.insert(0, str(TUTORIAL_DIR.resolve()))


Mounted at /content/drive


## What is trained, and how ratios are read out

In either part the three equal-prior class densities are denoted by $(\Pi_S,\Pi_P,\Pi_L)$.  A classifier trained with the single objective

$$
\mathcal L_{\rm CE}=-\mathbb E\log d_y(\vartheta,x)
$$

returns $(d_S,d_P,d_L)=\operatorname{softmax}(s_S,s_P,s_L)$. Equal class priors give

$$
r_P=\frac{\Pi_S}{\Pi_P}=\frac{d_S}{d_P},\qquad
r_L=\frac{\Pi_S}{\Pi_L}=\frac{d_S}{d_L}.
$$

Each ensemble member evaluates softmax in float64 and forms these probability quotients.  The implementation averages the ten positive quotient estimates arithmetically and never explicitly evaluates `exp(logit_difference)`.  Conditional masses $Z_P=\mathbb E_{q_P}[r_P]$ and $Z_L=\mathbb E_{q_L}[r_L]$, and the Bayes bridge, are estimated only after training.  They never enter a loss, validation score, or gradient.


In [2]:
import copy
import gc
import hashlib
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from scipy.special import logsumexp
from scipy.stats import multivariate_normal, norm, wasserstein_distance
from torch.utils.data import DataLoader, TensorDataset

from utils_dual_hnde import importance_tail_summary
from utils_hnpe import (
    sample_spline_flow,
    spline_flow_log_prob,
    train_spline_flow,
)
from utils_plotting import export_standalone_figure_script


SEED = 19092026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def set_torch_seed(seed):
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def _flow_log_prob(flow_pack, target, *, context=None, batch_size=65_536):
    return spline_flow_log_prob(
        flow_pack, target, context=context, batch_size=batch_size
    )


SMOKE_MODE = False
FAST_MODE = False
# This revision is diagnostics-only: never retrain a network implicitly.
REUSE_TRAINED_NETWORKS = True
LOAD_IF_AVAILABLE = REUSE_TRAINED_NETWORKS

if SMOKE_MODE:
    RUN_TAG = "smoke"
    N_FLOW, N_CLASS = 2_000, 3_000
    FLOW_EPOCHS, CLASS_EPOCHS, CLASSIFIER_ENSEMBLE_SIZE = 2, 4, 1
    N_FLOW_AUDIT_SAMPLES = 128
    N_BRIDGE_AUDIT_GROUPS, N_BRIDGE_AUDIT_INNER = 12, 8
    N_BRIDGE_AUDIT_MASS_INNER, N_BRIDGE_AUDIT_BANKS = 32, 1
elif FAST_MODE:
    RUN_TAG = "fast"
    N_FLOW, N_CLASS = 35_000, 80_000
    FLOW_EPOCHS, CLASS_EPOCHS, CLASSIFIER_ENSEMBLE_SIZE = 12, 250, 10
    N_FLOW_AUDIT_SAMPLES = 1_024
    N_BRIDGE_AUDIT_GROUPS, N_BRIDGE_AUDIT_INNER = 48, 12
    N_BRIDGE_AUDIT_MASS_INNER, N_BRIDGE_AUDIT_BANKS = 512, 2
else:
    RUN_TAG = "full"
    N_FLOW, N_CLASS = 250_000, 500_000
    FLOW_EPOCHS, CLASS_EPOCHS, CLASSIFIER_ENSEMBLE_SIZE = 50, 250, 10
    N_FLOW_AUDIT_SAMPLES = 16_384
    N_BRIDGE_AUDIT_GROUPS, N_BRIDGE_AUDIT_INNER = 64, 16
    N_BRIDGE_AUDIT_MASS_INNER, N_BRIDGE_AUDIT_BANKS = 2_048, 2

N_CALIBRATION_CONTEXTS = 20 if SMOKE_MODE else (200 if FAST_MODE else 2_000)
N_CALIBRATION_SAMPLES = 128 if SMOKE_MODE else (1_024 if FAST_MODE else 2_048)
N_JOINT_CALIBRATION_CONTEXTS = 12 if SMOKE_MODE else (100 if FAST_MODE else 500)
N_JOINT_CALIBRATION_SAMPLES = 128 if SMOKE_MODE else (512 if FAST_MODE else 1_024)
N_POSTERIOR_REFERENCE = (
    4_000 if SMOKE_MODE else (100_000 if FAST_MODE else 500_000)
)
N_NORMALIZATION_CHECK = 64 if SMOKE_MODE else (2_048 if FAST_MODE else 8_192)
N_NORMALIZATION_PATH = 64 if SMOKE_MODE else (2_048 if FAST_MODE else 8_192)
N_EVIDENCE_OUTER = 64 if SMOKE_MODE else (256 if FAST_MODE else 512)
EVIDENCE_Z_BUDGETS = (
    (16, 32, 64) if SMOKE_MODE
    else ((128, 512, 2_048) if FAST_MODE else (512, 2_048, 8_192, 20_000))
)
N_SELECTION_DRAWS = 128 if SMOKE_MODE else (2_048 if FAST_MODE else 8_192)

MODEL_DIR = Path("models_exercise9_multiclass_3D_v1") / RUN_TAG
FIGURE_SCRIPT_DIR = Path("exercise9_multiclass_3D_v1_figures_scripts") / RUN_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_SCRIPT_DIR.mkdir(parents=True, exist_ok=True)

if REUSE_TRAINED_NETWORKS:
    required_checkpoints = [
        MODEL_DIR / "q_p_marginal_mu_alpha_given_x.pt",
        MODEL_DIR / "q_lm_x_given_mu_alpha.pt",
        MODEL_DIR / "q_p_joint_mu_alpha_beta_given_x.pt",
        MODEL_DIR / "q_l_x_given_mu_alpha_beta.pt",
    ]
    for directory in (
        "part1_marginal_three_class_ce",
        "part2_joint_three_class_ce",
    ):
        required_checkpoints.extend(
            MODEL_DIR / directory / f"classifier.member_{index:02d}.pt"
            for index in range(CLASSIFIER_ENSEMBLE_SIZE)
        )
    missing_checkpoints = [
        path for path in required_checkpoints if not path.is_file()
    ]
    if missing_checkpoints:
        preview = "\n".join(f"  - {path}" for path in missing_checkpoints[:8])
        if len(missing_checkpoints) > 8:
            preview += f"\n  - ... and {len(missing_checkpoints) - 8} more"
        raise FileNotFoundError(
            "Diagnostics-only rerun requested, but trained checkpoints are "
            "missing. The notebook stopped before any training. Expected:\n"
            + preview
        )
    print(
        f"Diagnostics-only rerun: verified {len(required_checkpoints)} "
        "existing network checkpoints; training is disabled."
    )

# Exact model and optimizer configuration from Exercise_9_multiclass.
FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 10,
    "hidden_features": 512,
    "hidden_layers": 4,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 1.0e-3,
    "min_learning_rate": 1.0e-11,
    "validation_fraction": 0.2,
    "patience": 10,
    "gradient_clip": 5.0,
}

# Every member remains the same plain, uniform MLP; no regularization or auxiliary loss.
CLASS_MODEL_CONFIG = {
    "hidden_features": 1024 if not SMOKE_MODE else 128,
    "hidden_layers": 4 if not SMOKE_MODE else 2,
}
CLASS_TRAINING_CONFIG = {
    "batch_size": 1024 if not SMOKE_MODE else 256,
    "validation_batch_size": 8192 if not SMOKE_MODE else 512,
    "n_epochs": CLASS_EPOCHS,
    "learning_rate": 1.0e-4,
    "min_learning_rate": 1.0e-9,
    "lr_scheduler_factor": 0.1,
    "lr_scheduler_step_epochs": 40,
    "validation_fraction": 0.2,
    "patience": CLASS_EPOCHS,
    "minimum_improvement": 1.0e-6,
}

THREE_CE_LABEL = "marginal multiclass CE"
JOINT_CE_LABEL = "joint multiclass CE"
X_DIM, MARGINAL_DIM, THETA_DIM = 4, 2, 3
X_OBS = np.array([0.7775, 1.3147, 0.6293, -0.0449], dtype=float)


def export_exercise9_multiclass_figure(fig, script_name):
    path = export_standalone_figure_script(
        fig, script_name=script_name, output_dir=FIGURE_SCRIPT_DIR
    )
    png_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.png"
    pdf_path = FIGURE_SCRIPT_DIR / f"{Path(script_name).stem}.pdf"
    fig.savefig(png_path, dpi=220, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("Exported:", path, png_path, pdf_path)
    return path


print(
    f"mode={RUN_TAG}, N_flow={N_FLOW:,}, N_class={N_CLASS:,}, "
    f"classifier_ensemble={CLASSIFIER_ENSEMBLE_SIZE}"
)
print("Part I flow target: (mu, alpha); beta is marginalized through simulation")
print("Part II flow target: (mu, alpha, beta); all parameters are explicit")
print("Ratio model: unchanged plain MLP ensemble, equal-prior multiclass CE only")
print(
    f"Diagnostics: posterior bank={N_POSTERIOR_REFERENCE:,}, "
    f"normalization={N_NORMALIZATION_CHECK:,} draws/point, "
    f"selection={N_SELECTION_DRAWS:,} "
    "draws/point"
)


Using device: cuda
Diagnostics-only rerun: verified 24 existing network checkpoints; training is disabled.
mode=full, N_flow=250,000, N_class=500,000, classifier_ensemble=10
Part I flow target: (mu, alpha); beta is marginalized through simulation
Part II flow target: (mu, alpha, beta); all parameters are explicit
Ratio model: unchanged plain MLP ensemble, equal-prior multiclass CE only
Diagnostics: posterior bank=500,000, normalization=8,192 draws/point, selection=8,192 draws/point


## Four-dimensional Gaussian-mixture simulator and exact validation densities

The design prior is independent in the three coordinates:

$$
\rho_\mu=0.9\mathcal N(0,1.5^2)+0.1\mathcal N(0,4^2),\quad
\rho_\alpha=0.9\mathcal N(0,1^2)+0.1\mathcal N(0,3^2),\quad
\rho_\beta=0.9\mathcal N(0,0.8^2)+0.1\mathcal N(0,2.4^2).
$$

Let

$$
b(\mu)=(\mu,\ 0.72\mu^2,\ 0.8\cos\mu,\ 0.55\sin(0.8\mu)),
$$

with nuisance loadings $v_\alpha=(0.8,-0.4,0.3,0.5)$ and $v_\beta=(-0.35,0.55,0.65,-0.45)$.  Conditional on $\theta$, the simulator is a two-component diagonal Gaussian mixture with weights $(0.72,0.28)$; the second component has a fixed offset and slightly broader resolution.

When $\beta$ is hidden, each simulator component combines with each component of $\rho_\beta$.  Thus $p_m(x\mid\mu,\alpha)$ is an exact **four-component multivariate Gaussian mixture**.  The notebook evaluates that formula only in validation cells; neither flow nor classifier receives a density, score, component label, or analytic simulator quantity.


In [3]:
def _mixture_logpdf(values, core_sigma, broad_sigma):
    values = np.asarray(values, dtype=float)
    return logsumexp(
        np.stack([
            np.log(0.9) + norm.logpdf(values, 0.0, core_sigma),
            np.log(0.1) + norm.logpdf(values, 0.0, broad_sigma),
        ]),
        axis=0,
    )


def design_mu_logpdf(mu):
    return _mixture_logpdf(mu, 1.5, 4.0)


def design_alpha_logpdf(alpha):
    return _mixture_logpdf(alpha, 1.0, 3.0)


def design_beta_logpdf(beta):
    return _mixture_logpdf(beta, 0.8, 2.4)


def design_marginal_logpdf(phi):
    phi = np.atleast_2d(np.asarray(phi, dtype=float))
    return design_mu_logpdf(phi[:, 0]) + design_alpha_logpdf(phi[:, 1])


def design_logpdf(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    return design_marginal_logpdf(theta[:, :2]) + design_beta_logpdf(theta[:, 2])


def _sample_mixture(n, core_sigma, broad_sigma, rng):
    broad = rng.random(int(n)) < 0.1
    sigma = np.where(broad, broad_sigma, core_sigma)
    return rng.normal(0.0, sigma)


def sample_mu(n, rng):
    return _sample_mixture(n, 1.5, 4.0, rng).astype(np.float32)


def sample_alpha(n, rng):
    return _sample_mixture(n, 1.0, 3.0, rng).astype(np.float32)


def sample_beta(n, rng):
    return _sample_mixture(n, 0.8, 2.4, rng).astype(np.float32)


def sample_design(n, rng):
    return np.column_stack([
        sample_mu(n, rng), sample_alpha(n, rng), sample_beta(n, rng)
    ]).astype(np.float32)


SIMULATOR_WEIGHTS = np.array([0.72, 0.28], dtype=float)
SIMULATOR_OFFSETS = np.array([
    [0.0, 0.0, 0.0, 0.0],
    [0.45, -0.32, 0.28, -0.38],
], dtype=float)
SIMULATOR_SIGMAS = np.array([
    [0.95, 0.38, 0.30, 0.42],
    [1.10, 0.46, 0.36, 0.50],
], dtype=float)
ALPHA_LOADING = np.array([0.8, -0.4, 0.3, 0.5], dtype=float)
BETA_LOADING = np.array([-0.35, 0.55, 0.65, -0.45], dtype=float)


def simulator_base_mean(mu):
    mu = np.asarray(mu, dtype=float).ravel()
    return np.column_stack([
        mu,
        0.72 * mu**2,
        0.8 * np.cos(mu),
        0.55 * np.sin(0.8 * mu),
    ])


def simulator_mean(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    mu, alpha, beta = theta[:, 0], theta[:, 1], theta[:, 2]
    return (
        simulator_base_mean(mu)
        + alpha[:, None] * ALPHA_LOADING
        + beta[:, None] * BETA_LOADING
    )


def simulate(theta, rng):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    component = rng.choice(2, size=len(theta), p=SIMULATOR_WEIGHTS)
    mean = simulator_mean(theta) + SIMULATOR_OFFSETS[component]
    noise = rng.normal(size=mean.shape) * SIMULATOR_SIGMAS[component]
    return (mean + noise).astype(np.float32)


def log_likelihood(x, theta):
    """Exact two-Gaussian likelihood, used only for validation."""
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(theta) > 1:
        x = np.repeat(x, len(theta), axis=0)
    if len(x) != len(theta):
        raise ValueError("x and theta must have matching rows or one x row.")
    mean = simulator_mean(theta)
    terms = []
    for weight, offset, sigma in zip(
        SIMULATOR_WEIGHTS, SIMULATOR_OFFSETS, SIMULATOR_SIGMAS
    ):
        terms.append(
            np.log(weight) + np.sum(norm.logpdf(x, mean + offset, sigma), axis=1)
        )
    return logsumexp(np.stack(terms), axis=0)


_BETA_COMPONENTS = ((0.9, 0.8), (0.1, 2.4))
_MARGINAL_COMPONENTS = []
for simulator_weight, offset, sigma in zip(
    SIMULATOR_WEIGHTS, SIMULATOR_OFFSETS, SIMULATOR_SIGMAS
):
    for beta_weight, beta_sigma in _BETA_COMPONENTS:
        covariance = (
            np.diag(sigma**2)
            + beta_sigma**2 * np.outer(BETA_LOADING, BETA_LOADING)
        )
        _MARGINAL_COMPONENTS.append(
            (simulator_weight * beta_weight, offset, covariance)
        )


def marginal_log_likelihood(x, phi):
    """Exact four-Gaussian p_m(x|mu,alpha) after integrating beta."""
    phi = np.atleast_2d(np.asarray(phi, dtype=float))
    x = np.atleast_2d(np.asarray(x, dtype=float))
    if len(x) == 1 and len(phi) > 1:
        x = np.repeat(x, len(phi), axis=0)
    if len(x) != len(phi):
        raise ValueError("x and phi must have matching rows or one x row.")
    base = simulator_base_mean(phi[:, 0]) + phi[:, 1, None] * ALPHA_LOADING
    terms = []
    for weight, offset, covariance in _MARGINAL_COMPONENTS:
        residual = x - base - offset
        terms.append(
            np.log(weight)
            + np.atleast_1d(
                multivariate_normal.logpdf(
                    residual, mean=np.zeros(X_DIM), cov=covariance
                )
            )
        )
    return np.atleast_1d(logsumexp(np.stack(terms), axis=0))


def normalize_log_curve(log_density, grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(density, grid)
    return density / integral, shift + np.log(integral)


def normalize_log_surface(log_density, x_grid, y_grid):
    log_density = np.asarray(log_density, dtype=float)
    shift = float(np.max(log_density))
    density = np.exp(log_density - shift)
    integral = np.trapezoid(np.trapezoid(density, y_grid, axis=1), x_grid)
    return density / integral, shift + np.log(integral)


def integrated_absolute_error(reference, estimate, grid):
    return float(np.trapezoid(np.abs(reference - estimate), grid))


def surface_iae(reference, estimate, x_grid, y_grid):
    return float(
        np.trapezoid(
            np.trapezoid(np.abs(reference - estimate), y_grid, axis=1),
            x_grid,
        )
    )


def js_distance_discrete(reference, estimate, floor=1.0e-15):
    reference = np.asarray(reference, dtype=float).ravel() + floor
    estimate = np.asarray(estimate, dtype=float).ravel() + floor
    reference /= reference.sum()
    estimate /= estimate.sum()
    middle = 0.5 * (reference + estimate)
    divergence = 0.5 * np.sum(reference * np.log(reference / middle))
    divergence += 0.5 * np.sum(estimate * np.log(estimate / middle))
    return float(np.sqrt(max(0.0, divergence)))


print("Observed x:", X_OBS)
print(
    "Truth check, log p_m(x_obs|mu=1.2,alpha=-0.3):",
    marginal_log_likelihood(X_OBS, [[1.2, -0.3]])[0],
)


def sample_tail_enriched_design(n_samples, rng, tail_fraction=0.25):
    """Sample held-out bridge contexts with extra empirical tail coverage."""
    n_samples = int(n_samples)
    n_tail = min(n_samples, max(1, int(round(tail_fraction * n_samples))))
    bulk = sample_design(n_samples - n_tail, rng)
    pool = sample_design(max(1_024, 12 * n_tail), rng)
    center = np.median(pool, axis=0)
    mad = 1.4826 * np.median(np.abs(pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((pool - center) / scale), axis=1)
    tail_pool = np.flatnonzero(score >= np.quantile(score, 0.90))
    selected = rng.choice(tail_pool, size=n_tail, replace=len(tail_pool) < n_tail)
    combined = np.concatenate([bulk, pool[selected]], axis=0)
    return combined[rng.permutation(len(combined))].astype(np.float32)


Observed x: [ 0.7775  1.3147  0.6293 -0.0449]
Truth check, log p_m(x_obs|mu=1.2,alpha=-0.3): -2.046349009644205


## A plain sample-only multiclass ratio estimator

The classifier receives only sampled coordinates.  Inputs are standardized with the training-sample mean and standard deviation.  Full and fast modes use four identical `Linear(1024)` + `SiLU` layers; smoke mode uses two `Linear(128)` + `SiLU` layers solely for a quick code-path test.  Both end in three unconstrained logits.  There are no residual connections, normalization layers, dropout layers, bounded outputs, or auxiliary heads.

Each simulator/proposal group is assigned wholly to training or validation before its three rows are flattened. Thus paired rows cannot leak across the split. All classes occur exactly once per group, so their empirical priors are equal. Ten independent Adam fits minimize ordinary multiclass cross entropy for 250 epochs with batch size 1024. The learning rate is $10^{-4}$ for epochs 1--40 and is divided by ten every 40 epochs, reaching $10^{-9}$ for epochs 201--250. Patience spans the full schedule, and held-out multiclass CE alone selects each member's checkpoint.

Ratios are evaluated from float64 softmax probabilities.  No density, analytic score, simulator mean, or handcrafted residual is an input to the network.


In [4]:
class PlainMulticlassMLP(nn.Module):
    # Uniform Linear-SiLU stack followed by three unconstrained logits.

    def __init__(self, input_dim, n_classes, hidden_features, hidden_layers):
        super().__init__()
        layers = []
        width_in = int(input_dim)
        for _ in range(int(hidden_layers)):
            layers.extend([nn.Linear(width_in, int(hidden_features)), nn.SiLU()])
            width_in = int(hidden_features)
        layers.append(nn.Linear(width_in, int(n_classes)))
        self.network = nn.Sequential(*layers)
        self.input_dim = int(input_dim)
        self.n_classes = int(n_classes)

    def forward(self, values):
        return self.network(values)
def _assert_finite(name, values, ndim=None):
    values = np.asarray(values)
    if ndim is not None and values.ndim != ndim:
        raise ValueError(f"{name} must have ndim={ndim}; got {values.shape}.")
    if not np.isfinite(values).all():
        raise ValueError(f"{name} contains non-finite values.")
    return values

def _install_nflows_rqs_float64_retry():
    """Retry only a failed float32 inverse-RQS kernel in float64.

    For a monotone rational-quadratic spline the inverse discriminant is
    non-negative analytically.  nflows 0.14 evaluates it in float32, where a
    nearly double root can acquire a tiny negative value by cancellation.  We
    retry the *same* spline tensors in float64: no row is dropped, clipped, or
    resampled.  The original float64 assertion remains the hard guard.
    """
    import functools
    import importlib
    import inspect
    from importlib.metadata import version
    import warnings

    nflows_version = version("nflows")
    if nflows_version != "0.14":
        raise RuntimeError(
            "This audited numerical guard requires nflows==0.14; "
            f"found {nflows_version}."
        )
    module = importlib.import_module(
        "nflows.transforms.splines.rational_quadratic"
    )
    original = module.rational_quadratic_spline
    if getattr(original, "_exercise9_float64_retry", False):
        return original
    signature = inspect.signature(original)

    @functools.wraps(original)
    def guarded(*args, **kwargs):
        inputs_fast = kwargs.get("inputs", args[0] if args else None)
        inverse_fast = kwargs.get(
            "inverse", args[4] if len(args) > 4 else False
        )
        if (
            inverse_fast
            and torch.is_tensor(inputs_fast)
            and inputs_fast.dtype == torch.float32
        ):
            guarded._float32_inverse_call_count += 1
            guarded._float32_inverse_values += int(inputs_fast.numel())
        try:
            return original(*args, **kwargs)
        except AssertionError as error32:
            bound = signature.bind(*args, **kwargs)
            inputs = bound.arguments["inputs"]
            inverse = bound.arguments.get(
                "inverse", signature.parameters["inverse"].default
            )
            if not inverse or inputs.dtype != torch.float32:
                raise

            floating = [
                value for value in (*args, *kwargs.values())
                if torch.is_tensor(value) and value.is_floating_point()
            ]
            if any(not bool(torch.isfinite(value).all()) for value in floating):
                raise FloatingPointError(
                    "Non-finite tensor reached the inverse RQS; refusing the "
                    "precision retry."
                ) from error32

            def to_float64(value):
                if torch.is_tensor(value) and value.is_floating_point():
                    return value.to(dtype=torch.float64)
                return value

            try:
                outputs64, logdet64 = original(
                    *(to_float64(value) for value in args),
                    **{
                        name: to_float64(value)
                        for name, value in kwargs.items()
                    },
                )
            except AssertionError as error64:
                raise RuntimeError(
                    "The inverse-RQS discriminant also failed in float64. "
                    "Refusing to clip or resample; retrain this flow."
                ) from error64
            if not (
                bool(torch.isfinite(outputs64).all())
                and bool(torch.isfinite(logdet64).all())
            ):
                raise FloatingPointError(
                    "The float64 inverse-RQS retry returned non-finite values."
                ) from error32

            guarded._float64_retry_count += 1
            guarded._float64_retry_values += int(inputs.numel())
            if guarded._float64_retry_count == 1:
                warnings.warn(
                    "nflows float32 inverse-RQS cancellation: retrying the "
                    "same spline call in float64.",
                    RuntimeWarning,
                    stacklevel=2,
                )
            outputs = outputs64.to(dtype=inputs.dtype)
            logdet = logdet64.to(dtype=inputs.dtype)
            if not (
                bool(torch.isfinite(outputs).all())
                and bool(torch.isfinite(logdet).all())
            ):
                raise FloatingPointError(
                    "Casting the inverse-RQS retry back to float32 "
                    "produced non-finite values."
                ) from error32
            return outputs, logdet

    guarded._exercise9_float64_retry = True
    guarded._float64_retry_count = 0
    guarded._float64_retry_values = 0
    guarded._float32_inverse_call_count = 0
    guarded._float32_inverse_values = 0
    guarded._float32_original = original
    module.rational_quadratic_spline = guarded

    # nflows 0.14's linear-tail helper resolves the module global above.  The
    # aliases cover any direct bounded-spline call without touching package
    # source or a checkpoint.
    importlib.import_module(
        "nflows.transforms.splines"
    ).rational_quadratic_spline = guarded
    importlib.import_module(
        "nflows.transforms.autoregressive"
    ).rational_quadratic_spline = guarded
    linear_tail = importlib.import_module(
        "nflows.transforms.splines"
    ).unconstrained_rational_quadratic_spline
    if linear_tail.__globals__.get("rational_quadratic_spline") is not guarded:
        raise RuntimeError(
            "The nflows 0.14 linear-tail inverse did not bind to the "
            "audited RQS guard."
        )
    return guarded


RQS_NUMERIC_GUARD = _install_nflows_rqs_float64_retry()


def _rqs_retry_count():
    return int(getattr(RQS_NUMERIC_GUARD, "_float64_retry_count", 0))


def _rqs_inverse_call_count():
    return int(
        getattr(RQS_NUMERIC_GUARD, "_float32_inverse_call_count", 0)
    )


def _draw_conditional(flow_pack, contexts, n_samples, seed, allocation=None):
    # allocation is retained as a compatibility no-op for old call sites;
    # This notebook uses one proposal flow, hence no component labels to allocate.
    del allocation
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    n_samples = int(n_samples)
    n_features = int(flow_pack["config"]["n_features"])
    set_torch_seed(seed)
    draws = sample_spline_flow(
        flow_pack, n_samples, context=contexts, batch_size=16_384
    )
    draws = np.asarray(draws, dtype=np.float32)
    if len(contexts) == 1:
        draws = draws[None, :, :]
    expected = (len(contexts), n_samples, n_features)
    if draws.shape != expected:
        raise RuntimeError(
            f"Unexpected conditional sample shape {draws.shape}; expected {expected}."
        )
    return _assert_finite("conditional flow draws", draws, ndim=3)



def _audit_context_subset(flow_pack, contexts, seed, n_contexts=2_048):
    contexts = _assert_finite("flow audit contexts", contexts, ndim=2).astype(
        np.float32
    )
    if len(contexts) < n_contexts:
        raise ValueError("The flow audit needs at least n_contexts rows.")
    scaler = flow_pack["context_scaler"]
    standardized = (contexts - scaler.mean) / scaler.std
    extremeness = np.max(np.abs(standardized), axis=1)
    n_tail = min(512, n_contexts // 4)
    tail_index = np.argpartition(extremeness, -n_tail)[-n_tail:]
    available = np.setdiff1d(
        np.arange(len(contexts)), tail_index, assume_unique=False
    )
    rng = np.random.default_rng(int(seed))
    typical_index = rng.choice(
        available, size=n_contexts - n_tail, replace=False
    )
    return contexts[np.concatenate([typical_index, tail_index])]


def _fit_classifier_transform(points):
    points = np.asarray(points, dtype=np.float32)
    center = points.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = points.std(axis=0, dtype=np.float64).astype(np.float32)
    scale = np.where(scale > 1.0e-6, scale, 1.0).astype(np.float32)
    return center, scale


def _transform_classifier_points(points, center, scale):
    return ((np.asarray(points, dtype=np.float32) - center) / scale).astype(np.float32)


def _make_model(input_dim, n_classes):
    return PlainMulticlassMLP(
        input_dim=input_dim,
        n_classes=n_classes,
        **CLASS_MODEL_CONFIG,
    ).to(device)


def _multiclass_fingerprint(class_points, *, n_classes, seed_base):
    digest = hashlib.sha256()
    configuration = json.dumps(
        {
            "n_classes": int(n_classes),
            "seed_base": int(seed_base),
            "classifier_count": int(CLASSIFIER_ENSEMBLE_SIZE),
            "model": CLASS_MODEL_CONFIG,
            "training": CLASS_TRAINING_CONFIG,
            "loss": "equal_prior_multiclass_ce_only_v9_classifier_ensemble",
            "input_transform": "ordinary_standardization_v1",
        },
        sort_keys=True,
        separators=(",", ":"),
    )
    digest.update(configuration.encode("utf-8"))
    array = np.ascontiguousarray(class_points)
    digest.update(str(array.shape).encode("ascii"))
    digest.update(str(array.dtype).encode("ascii"))
    digest.update(array.view(np.uint8))
    return digest.hexdigest()


def _validation_ce(model, points, labels):
    model.eval()
    batch_size = int(CLASS_TRAINING_CONFIG["validation_batch_size"])
    loss_sum, count = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(points), batch_size):
            stop = start + batch_size
            batch = points[start:stop].to(device)
            target = labels[start:stop].to(device)
            loss = F.cross_entropy(model(batch), target)
            loss_sum += float(loss.cpu()) * len(batch)
            count += len(batch)
    return loss_sum / max(1, count)


def train_multiclass_classifier(
    class_points,
    *,
    n_classes,
    checkpoint_dir,
    seed_base,
):
    # Train equal-prior multiclass cross entropy, and no other term.
    class_points = _assert_finite(
        "class_points", class_points, ndim=3
    ).astype(np.float32)
    if class_points.shape[1] != int(n_classes):
        raise ValueError("One row per class is required in every group.")

    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = _multiclass_fingerprint(
        class_points, n_classes=n_classes, seed_base=seed_base
    )

    split_rng = np.random.default_rng(SEED + 700 + int(n_classes))
    order = split_rng.permutation(len(class_points))
    n_validation = max(
        1, int(CLASS_TRAINING_CONFIG["validation_fraction"] * len(order))
    )
    validation_index, training_index = order[:n_validation], order[n_validation:]

    training_flat = class_points[training_index].reshape(
        -1, class_points.shape[-1]
    )
    validation_flat = class_points[validation_index].reshape(
        -1, class_points.shape[-1]
    )
    center, scale = _fit_classifier_transform(training_flat)
    training_tensor = torch.as_tensor(
        _transform_classifier_points(training_flat, center, scale),
        dtype=torch.float32,
    )
    validation_tensor = torch.as_tensor(
        _transform_classifier_points(validation_flat, center, scale),
        dtype=torch.float32,
    )
    training_labels = torch.arange(int(n_classes), dtype=torch.long).repeat(
        len(training_index)
    )
    validation_labels = torch.arange(int(n_classes), dtype=torch.long).repeat(
        len(validation_index)
    )
    training_dataset = TensorDataset(training_tensor, training_labels)

    classifier_packs = []
    for classifier_index in range(CLASSIFIER_ENSEMBLE_SIZE):
        classifier_seed = int(seed_base + classifier_index)
        checkpoint = checkpoint_dir / f"classifier.member_{classifier_index:02d}.pt"
        if LOAD_IF_AVAILABLE and checkpoint.exists():
            try:
                saved = torch.load(
                    checkpoint, map_location=device, weights_only=False
                )
            except TypeError:
                saved = torch.load(checkpoint, map_location=device)
            if (
                saved.get("fingerprint") != fingerprint
                or int(saved.get("classifier_index", -1)) != classifier_index
                or int(saved.get("classifier_seed", -1)) != classifier_seed
            ):
                raise RuntimeError(
                    f"Checkpoint {checkpoint} belongs to a different CE experiment."
                )
            model = _make_model(class_points.shape[-1], n_classes)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            classifier_packs.append(
                {
                    "model": model,
                    "center": np.asarray(saved["center"], dtype=np.float32),
                    "scale": np.asarray(saved["scale"], dtype=np.float32),
                    "history": saved.get("history", {}),
                    "checkpoint": checkpoint,
                    "classifier_index": classifier_index,
                    "classifier_seed": classifier_seed,
                }
            )
            print("Loaded", checkpoint)
            continue

        set_torch_seed(classifier_seed)
        generator = torch.Generator().manual_seed(classifier_seed + 120_000)
        loader = DataLoader(
            training_dataset,
            batch_size=int(CLASS_TRAINING_CONFIG["batch_size"]),
            shuffle=True,
            generator=generator,
        )
        model = _make_model(class_points.shape[-1], n_classes)
        if classifier_index == 0:
            parameter_count = sum(p.numel() for p in model.parameters())
            print(f"Plain {n_classes}-class MLP parameters/member: {parameter_count:,}")
            print(f"Classifier ensemble members: {CLASSIFIER_ENSEMBLE_SIZE}")
            print("Optimized objective: equal-prior multiclass CE only")
        optimizer = torch.optim.Adam(
            model.parameters(), lr=float(CLASS_TRAINING_CONFIG["learning_rate"])
        )
        history = {"train_ce": [], "validation_ce": [], "learning_rate": []}
        best_state, best_value, best_epoch, stale = None, math.inf, 0, 0

        for epoch in range(int(CLASS_TRAINING_CONFIG["n_epochs"])):
            learning_rate = max(
                float(CLASS_TRAINING_CONFIG["min_learning_rate"]),
                float(CLASS_TRAINING_CONFIG["learning_rate"])
                * float(CLASS_TRAINING_CONFIG["lr_scheduler_factor"])
                ** (
                    epoch
                    // int(CLASS_TRAINING_CONFIG["lr_scheduler_step_epochs"])
                ),
            )
            for parameter_group in optimizer.param_groups:
                parameter_group["lr"] = learning_rate
            model.train()
            train_sum, train_count = 0.0, 0
            for batch, labels in loader:
                batch = batch.to(device)
                labels = labels.to(device)
                objective = F.cross_entropy(model(batch), labels)
                if not torch.isfinite(objective):
                    raise FloatingPointError("Non-finite multiclass CE.")
                optimizer.zero_grad(set_to_none=True)
                objective.backward()
                optimizer.step()
                train_sum += float(objective.detach().cpu()) * len(batch)
                train_count += len(batch)

            train_ce = train_sum / max(1, train_count)
            validation_ce = _validation_ce(
                model, validation_tensor, validation_labels
            )
            history["train_ce"].append(train_ce)
            history["validation_ce"].append(validation_ce)
            history["learning_rate"].append(
                float(optimizer.param_groups[0]["lr"])
            )
            if validation_ce < best_value - float(
                CLASS_TRAINING_CONFIG["minimum_improvement"]
            ):
                best_value = validation_ce
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch + 1
                stale = 0
            else:
                stale += 1
            if (
                epoch == 0
                or (epoch + 1)
                % int(CLASS_TRAINING_CONFIG["lr_scheduler_step_epochs"])
                == 0
                or epoch + 1 == int(CLASS_TRAINING_CONFIG["n_epochs"])
            ):
                print(
                    f"classifier {classifier_index + 1:02d}/"
                    f"{CLASSIFIER_ENSEMBLE_SIZE:02d}, epoch {epoch + 1:3d}: "
                    f"train CE={train_ce:.6f}, validation CE={validation_ce:.6f}, "
                    f"lr={learning_rate:.1e}"
                )
            if stale >= int(CLASS_TRAINING_CONFIG["patience"]):
                print(f"classifier: early stopping after {epoch + 1} epochs")
                break

        if best_state is None:
            raise RuntimeError("No finite CE checkpoint was produced.")
        model.load_state_dict(best_state)
        model.eval()
        history["selected"] = [
            {"epoch": best_epoch, "validation_ce": best_value}
        ]
        history["optimized_terms"] = ["multiclass_ce"]
        torch.save(
            {
                "state_dict": model.state_dict(),
                "center": center,
                "scale": scale,
                "history": history,
                "n_classes": int(n_classes),
                "input_dim": int(class_points.shape[-1]),
                "model_config": CLASS_MODEL_CONFIG,
                "fingerprint": fingerprint,
                "classifier_index": classifier_index,
                "classifier_seed": classifier_seed,
            },
            checkpoint,
        )
        classifier_packs.append(
            {
                "model": model,
                "center": center,
                "scale": scale,
                "history": history,
                "checkpoint": checkpoint,
                "classifier_index": classifier_index,
                "classifier_seed": classifier_seed,
            }
        )
    return classifier_packs


@torch.no_grad()
def predict_class_probabilities(classifier_packs, points, batch_size=65_536):
    """Proxy probabilities encoding arithmetic means of member ratios.

    Every downstream correction uses class 0 as its numerator.  For each
    alternative class j, this routine first computes the member-wise
    positive softmax quotient d_0/d_j and then averages those quotients.
    The returned normalized proxy has exactly those averaged quotients,
    so the existing probability-ratio interface remains unchanged.
    """
    if not classifier_packs:
        raise RuntimeError("At least one classifier is required.")
    points = _assert_finite("prediction points", points)
    original_shape = points.shape[:-1]
    flat = points.reshape(-1, points.shape[-1]).astype(np.float32)
    tiny = np.finfo(np.float64).tiny
    chunks = []
    for start in range(0, len(flat), int(batch_size)):
        stop = start + int(batch_size)
        ratio_mean = None
        for pack in classifier_packs:
            transformed = _transform_classifier_points(
                flat[start:stop], pack["center"], pack["scale"]
            )
            tensor = torch.as_tensor(
                transformed, dtype=torch.float32, device=device
            )
            probabilities = (
                torch.softmax(pack["model"](tensor).to(torch.float64), dim=1)
                .detach()
                .cpu()
                .numpy()
            )
            member_ratios = probabilities[:, :1] / np.maximum(
                probabilities[:, 1:], tiny
            )
            if ratio_mean is None:
                ratio_mean = member_ratios / float(len(classifier_packs))
            else:
                ratio_mean += member_ratios / float(len(classifier_packs))
        if ratio_mean is None or not np.isfinite(ratio_mean).all():
            raise FloatingPointError("Classifier ensemble returned non-finite ratios.")
        scores = np.concatenate(
            [np.ones((len(ratio_mean), 1)), 1.0 / np.maximum(ratio_mean, tiny)],
            axis=1,
        )
        scores /= np.max(scores, axis=1, keepdims=True)
        chunks.append(scores / scores.sum(axis=1, keepdims=True))
    probabilities = np.concatenate(chunks, axis=0)
    return probabilities.reshape(*original_shape, probabilities.shape[-1])


@torch.no_grad()
def predict_class_log_probabilities(classifier_packs, points, batch_size=65_536):
    points = _assert_finite("log-probability points", points)
    probabilities = predict_class_probabilities(
        classifier_packs, points, batch_size=batch_size
    )
    log_probabilities = np.log(
        np.maximum(probabilities, np.finfo(np.float64).tiny)
    )
    if not np.isfinite(log_probabilities).all():
        raise FloatingPointError("Classifier ensemble returned non-finite log ratios.")
    return log_probabilities


def class_probability_ratio(probabilities, numerator, denominator):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    denominator_probability = np.maximum(
        probabilities[..., int(denominator)], np.finfo(np.float64).tiny
    )
    return probabilities[..., int(numerator)] / denominator_probability


def class_log_ratio(log_probabilities, numerator, denominator):
    log_probabilities = np.asarray(log_probabilities, dtype=np.float64)
    return (
        log_probabilities[..., int(numerator)]
        - log_probabilities[..., int(denominator)]
    )


def normalized_probability_ratios(ratios, axis=-1):
    ratios = np.asarray(ratios, dtype=np.float64)
    total = np.sum(ratios, axis=axis, keepdims=True)
    if not np.all(np.isfinite(total)) or np.any(total <= 0.0):
        raise FloatingPointError("Invalid softmax-probability ratio mass.")
    return ratios / total


def normalized_log_weights(log_weights):
    # Stable normalization for generic density log weights.
    tensor = torch.as_tensor(log_weights, dtype=torch.float64)
    return torch.softmax(tensor, dim=0).cpu().numpy()


def probability_floor_fraction(probabilities, class_index, threshold=1.0e-12):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    return float(np.mean(probabilities[..., int(class_index)] < threshold))
def select_empirical_context_slices(context_pool):
    context_pool = _assert_finite("context pool", context_pool, ndim=2).astype(np.float32)
    center = np.median(context_pool, axis=0)
    mad = 1.4826 * np.median(np.abs(context_pool - center), axis=0)
    scale = np.where(mad > 1.0e-6, mad, context_pool.std(axis=0))
    scale = np.where(scale > 1.0e-6, scale, 1.0)
    score = np.max(np.abs((context_pool - center) / scale), axis=1)
    probabilities = np.array([0.50, 0.90, 0.97, 0.99, 0.995, 0.999, 0.9998, 1.0])
    targets = np.quantile(score, probabilities)
    indices = np.array([int(np.argmin(np.abs(score - value))) for value in targets])
    return context_pool[indices], score[indices], probabilities


def sample_only_flow_audit(flow_pack, contexts, truth_draws, scores, probabilities, name, seed):
    truth_draws = _assert_finite("simulator audit draws", truth_draws, ndim=3)
    flow_draws = _draw_conditional(flow_pack, contexts, truth_draws.shape[1], seed)
    rows = []
    quantiles = np.array([0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999])
    for index, (truth, learned) in enumerate(zip(truth_draws, flow_draws)):
        truth_scale = np.maximum(truth.std(axis=0), 1.0e-6)
        coordinate_w1 = np.array([
            wasserstein_distance(truth[:, feature], learned[:, feature])
            for feature in range(truth.shape[1])
        ]) / truth_scale
        mean_error = np.abs(learned.mean(axis=0) - truth.mean(axis=0)) / truth_scale
        quantile_error = np.max(np.abs(
            np.quantile(learned, quantiles, axis=0)
            - np.quantile(truth, quantiles, axis=0)
        ) / truth_scale, axis=0)

        projection_rng = np.random.default_rng(seed + 10_000 + index)
        directions = projection_rng.normal(size=(32, truth.shape[1]))
        directions /= np.linalg.norm(directions, axis=1, keepdims=True)
        truth_projection = truth @ directions.T
        learned_projection = learned @ directions.T
        projection_scale = np.maximum(truth_projection.std(axis=0), 1.0e-6)
        sliced_w1 = np.array([
            wasserstein_distance(truth_projection[:, direction], learned_projection[:, direction])
            for direction in range(len(directions))
        ]) / projection_scale

        truth_correlation = np.corrcoef(truth, rowvar=False)
        learned_correlation = np.corrcoef(learned, rowvar=False)
        correlation_error = np.max(np.abs(truth_correlation - learned_correlation))
        rows.append({
            "flow": name,
            "context quantile": float(probabilities[index]),
            "context score": float(scores[index]),
            "max coordinate W1 / truth sigma": float(np.max(coordinate_w1)),
            "q95 sliced W1 / truth sigma": float(np.quantile(sliced_w1, 0.95)),
            "max mean error / truth sigma": float(np.max(mean_error)),
            "max tail-quantile error / truth sigma": float(np.max(quantile_error)),
            "max correlation error": float(correlation_error),
        })
    result = pd.DataFrame(rows)
    if not np.isfinite(result.select_dtypes(include=[np.number])).all().all():
        raise FloatingPointError(f"{name}: non-finite sample-only flow audit.")
    print(
        f"{name} sample-only joint-tail audit: worst q95 sliced W1="
        f"{result['q95 sliced W1 / truth sigma'].max():.3f}."
    )
    return result


# Part I — one nuisance marginalized, two parameters retained

We hide only $\beta$.  Independent simulator samples train

$$
q_P^m(\mu,\alpha\mid x),\qquad q_L^m(x\mid\mu,\alpha).
$$

Unlike the scalar Part I in `Exercise_9_multiclass.ipynb`, both targets here are multivariate.  In particular, $q_P^m$ is a two-dimensional coupling flow: the unchanged ten coupling layers alternate which of $(\mu,\alpha)$ is transformed.  The classifier sees $(\mu,\alpha,x_1,\ldots,x_4)$ and no value or label for $\beta$.

The three balanced classes remain simulator joint, posterior-proposal joint, and likelihood-proposal joint.  Only samples are supplied to the flows and classifier.


In [5]:
# Independent simulations for the two frozen marginalized proposals.
rng = np.random.default_rng(SEED + 10)
theta_qp = sample_design(N_FLOW, rng)
x_qp = simulate(theta_qp, rng)
set_torch_seed(SEED + 11)
q_p = train_spline_flow(
    theta_qp[:, :MARGINAL_DIM],
    context=x_qp,
    checkpoint=MODEL_DIR / "q_p_marginal_mu_alpha_given_x.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 11,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qp, x_qp

rng = np.random.default_rng(SEED + 20)
theta_qlm = sample_design(N_FLOW, rng)
x_qlm = simulate(theta_qlm, rng)
set_torch_seed(SEED + 21)
q_lm = train_spline_flow(
    x_qlm,
    context=theta_qlm[:, :MARGINAL_DIM],
    checkpoint=MODEL_DIR / "q_lm_x_given_mu_alpha.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 21,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qlm, x_qlm

# Sample-only q_L^m audit at central through empirically extreme (mu,alpha) ranks.
rng = np.random.default_rng(SEED + 22)
qlm_context_pool = sample_design(
    50_000 if not SMOKE_MODE else 2_000, rng
)[:, :MARGINAL_DIM]
qlm_audit_contexts, qlm_audit_scores, qlm_audit_probabilities = (
    select_empirical_context_slices(qlm_context_pool)
)
n_audit_contexts = len(qlm_audit_contexts)
phi_truth = np.repeat(qlm_audit_contexts, N_FLOW_AUDIT_SAMPLES, axis=0)
beta_truth = sample_beta(n_audit_contexts * N_FLOW_AUDIT_SAMPLES, rng)
qlm_truth_draws = simulate(
    np.column_stack([phi_truth, beta_truth]), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, X_DIM)
qlm_tail_audit = sample_only_flow_audit(
    q_lm,
    qlm_audit_contexts,
    qlm_truth_draws,
    qlm_audit_scores,
    qlm_audit_probabilities,
    r"$q_L^m(x\mid\mu,\alpha)$",
    SEED + 23,
)
display(qlm_tail_audit.style.format(precision=4))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loaded spline flow with matching data fingerprint from models_exercise9_multiclass_3D_v1/full/q_p_marginal_mu_alpha_given_x.pt
Loaded spline flow with matching data fingerprint from models_exercise9_multiclass_3D_v1/full/q_lm_x_given_mu_alpha.pt
$q_L^m(x\mid\mu,\alpha)$ sample-only joint-tail audit: worst q95 sliced W1=223.871.


,flow,context quantile,context score,max coordinate W1 / truth sigma,q95 sliced W1 / truth sigma,max mean error / truth sigma,max tail-quantile error / truth sigma,max correlation error
0,"$q_L^m(x\mid\mu,\alpha)$",0.5000,1.0685,6.1737,5.2861,1.1665,23.6276,0.6056
1,"$q_L^m(x\mid\mu,\alpha)$",0.9000,2.2731,6.3224,6.2470,2.5023,23.3717,0.5811
2,"$q_L^m(x\mid\mu,\alpha)$",0.9700,3.7902,33.7631,37.2772,33.7632,56.7381,0.5989
3,"$q_L^m(x\mid\mu,\alpha)$",0.9900,5.1505,63.8670,57.9973,63.8669,86.0591,0.5274
4,"$q_L^m(x\mid\mu,\alpha)$",0.9950,5.8581,84.3008,69.3855,84.3006,105.8253,0.5900
5,"$q_L^m(x\mid\mu,\alpha)$",0.9990,7.6507,8.4513,7.4056,6.8859,29.1580,0.5712
6,"$q_L^m(x\mid\mu,\alpha)$",0.9998,8.8680,198.5026,223.8715,198.5028,221.3012,0.6415
7,"$q_L^m(x\mid\mu,\alpha)$",1.0000,10.4459,10.4974,12.2964,9.6999,32.8507,0.6430


### Preflight: multivariate sampling before building classifier classes

Both Part-I flows now use ordinary alternating coupling masks.  Before a large inverse draw constructs the classifier classes, we check finite log densities, seeded reproducibility, target dimensionality, and representative central/tail contexts.  No density-aware repair, row dropping, clipping, or resampling is permitted.


In [6]:
def audit_vector_flow(flow_pack, contexts, expected_features, name, seed):
    if int(flow_pack["config"]["n_features"]) != int(expected_features):
        raise RuntimeError(
            f"{name}: expected {expected_features} target features; "
            f"found {flow_pack['config']['n_features']}."
        )
    contexts = np.atleast_2d(np.asarray(contexts, dtype=np.float32))
    retry_before = _rqs_retry_count()
    first = _draw_conditional(flow_pack, contexts, 8, seed)
    second = _draw_conditional(flow_pack, contexts, 8, seed)
    if not np.array_equal(first, second):
        raise RuntimeError(f"{name}: repeated seeded draws are not identical.")
    repeated_context = np.repeat(contexts[:, None, :], 8, axis=1).reshape(
        -1, contexts.shape[1]
    )
    log_density = _flow_log_prob(
        flow_pack,
        first.reshape(-1, expected_features),
        context=repeated_context,
    )
    if not np.isfinite(log_density).all():
        raise FloatingPointError(f"{name}: non-finite density at its own draws.")
    retry_delta = _rqs_retry_count() - retry_before
    print(
        f"{name} vector-flow audit passed: target shape={first.shape}, "
        f"finite log-density range=({log_density.min():.2f}, "
        f"{log_density.max():.2f}), retried RQS kernels={retry_delta}."
    )


rng = np.random.default_rng(SEED + 90)
theta_qp_audit = sample_design(2_048 if not SMOKE_MODE else 128, rng)
tail_theta = np.array([
    [-8.0, 0.0, 0.0], [8.0, 0.0, 0.0],
    [0.0, -6.0, 0.0], [0.0, 6.0, 0.0],
], dtype=np.float32)
x_qp_audit = np.concatenate([
    simulate(theta_qp_audit[:12], rng), simulate(tail_theta, rng)
])
audit_vector_flow(
    q_p, x_qp_audit, MARGINAL_DIM, r"$q_P^m(\mu,\alpha\mid x)$", SEED + 91
)
audit_vector_flow(
    q_lm,
    theta_qp_audit[:16, :MARGINAL_DIM],
    X_DIM,
    r"$q_L^m(x\mid\mu,\alpha)$",
    SEED + 92,
)
del theta_qp_audit, tail_theta, x_qp_audit


$q_P^m(\mu,\alpha\mid x)$ vector-flow audit passed: target shape=(16, 8, 2), finite log-density range=(-12.34, 1.37), retried RQS kernels=0.
$q_L^m(x\mid\mu,\alpha)$ vector-flow audit passed: target shape=(16, 8, 4), finite log-density range=(-13.06, -1.32), retried RQS kernels=0.


In [7]:
def build_three_class_groups(n_groups, q_p, q_lm, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    phi_s = theta_s[:, :MARGINAL_DIM]
    x_s = simulate(theta_s, rng)
    phi_p = _draw_conditional(q_p, x_s, 1, seed + 1)[:, 0, :]
    x_l = _draw_conditional(q_lm, phi_s, 1, seed + 2)[:, 0, :]
    points_s = np.column_stack([phi_s, x_s])
    points_p = np.column_stack([phi_p, x_s])
    points_l = np.column_stack([phi_s, x_l])
    groups = np.stack([points_s, points_p, points_l], axis=1).astype(np.float32)
    return _assert_finite("three-class groups", groups, ndim=3)


three_retry_before = _rqs_retry_count()
three_class_groups = build_three_class_groups(N_CLASS, q_p, q_lm, SEED + 100)
print("Marginal three-class grouped tensor:", three_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried during class construction:",
    _rqs_retry_count() - three_retry_before,
)


Marginal three-class grouped tensor: (500000, 3, 6)
Inverse-RQS kernel calls retried during class construction: 0


## Post-training conditional normalization and bridge check

The pure-CE residuals need not integrate to one at finite capacity.  After the classifier is frozen we estimate

$$
Z_P^m(x)=\mathbb E_{q_P^m}[r_P],\qquad
Z_L^m(\mu,\alpha)=\mathbb E_{q_L^m}[r_L]
$$

which equal one at the population optimum.  A low-statistics estimate of $Z_L^m(\mu,\alpha)$ is a noisy function of the parameters and can create artificial posterior structure.  We therefore use the smooth CE-corrected likelihood **before** $Z_L^m$ as the primary closure and estimate conditional masses only with high statistics on held-out points and sparse paths.  The resulting Bayes bridge must be constant in $(\mu,\alpha)$ for fixed $x$.  Normalization and the bridge remain post-training consistency checks only.  The exact four-Gaussian marginalized likelihood is never used to construct a class or select a checkpoint.


In [8]:
def build_three_bridge_bundle(n_groups, n_inner, n_mass_inner, q_p, q_lm, seed):
    if int(n_inner) < 2:
        raise ValueError("Bridge variance requires at least two draws per anchor.")
    rng = np.random.default_rng(seed)
    theta_anchor = sample_tail_enriched_design(n_groups, rng)
    x_anchor = simulate(theta_anchor, rng)
    phi = _draw_conditional(q_p, x_anchor, n_inner, seed + 1)
    x_repeat = np.repeat(x_anchor[:, None, :], n_inner, axis=1)
    points = np.concatenate([phi, x_repeat], axis=2).astype(np.float32)
    flat_phi = phi.reshape(-1, MARGINAL_DIM)
    flat_x = x_repeat.reshape(-1, X_DIM)
    bridge_base = (
        design_marginal_logpdf(flat_phi)
        + _flow_log_prob(q_lm, flat_x, context=flat_phi)
        - _flow_log_prob(q_p, flat_phi, context=flat_x)
    ).reshape(n_groups, n_inner).astype(np.float32)

    x_zl = _draw_conditional(q_lm, flat_phi, n_mass_inner, seed + 2)
    phi_zl = np.repeat(flat_phi[:, None, :], n_mass_inner, axis=1)
    bridge_zl_points = np.concatenate([phi_zl, x_zl], axis=2).reshape(
        n_groups, n_inner, n_mass_inner, MARGINAL_DIM + X_DIM
    ).astype(np.float32)
    return {
        "bridge_points": _assert_finite("marginal bridge points", points),
        "bridge_base": _assert_finite("marginal bridge base", bridge_base),
        "bridge_zl_points": _assert_finite(
            "marginal bridge ZL points", bridge_zl_points
        ),
    }


## Pure-CE three-class training

The loss, architecture, ten independent ensemble members, group-wise split, batch size, 250 epochs, and stepped $10^{-4}\!\to10^{-9}$ learning-rate schedule are unchanged from `Exercise_9_multiclass.ipynb`.


In [9]:
three_ce = train_multiclass_classifier(
    three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "part1_marginal_three_class_ce",
    seed_base=SEED + 400,
)
del three_class_groups
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


RuntimeError: Checkpoint models_exercise9_multiclass_3D_v1/full/part1_marginal_three_class_ce/classifier.member_00.pt belongs to a different CE experiment.

## Independent closure of the one-nuisance-marginalized experiment

The exact four-component Gaussian mixture supplies an external two-dimensional posterior reference.  We compare both corrected routes, their uncorrected flow proposals, high-statistics post-training mass checks, the bridge, and ratio tails.  A pointwise Monte Carlo $Z_L^m$ is deliberately not divided into the dense posterior surface.  No analytic value is reused in training.


In [ ]:
def three_zp(classifier_packs, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    phi = _draw_conditional(q_p, x_values, n_reference, seed)
    points = np.concatenate(
        [phi, np.repeat(x_values[:, None, :], n_reference, axis=1)], axis=2
    )
    probabilities = predict_class_probabilities(classifier_packs, points)
    return np.mean(class_probability_ratio(probabilities, 0, 1), axis=1)


def three_log_zp(classifier_packs, x_values, n_reference, seed):
    return np.log(three_zp(classifier_packs, x_values, n_reference, seed))


def three_zl(classifier_packs, phi_values, n_reference, seed):
    phi_values = np.atleast_2d(np.asarray(phi_values, dtype=np.float32))
    context_batch = max(1, 65_536 // int(n_reference))
    masses = []
    for start in range(0, len(phi_values), context_batch):
        phi_chunk = phi_values[start : start + context_batch]
        x = _draw_conditional(q_lm, phi_chunk, n_reference, seed + start)
        points = np.concatenate(
            [np.repeat(phi_chunk[:, None, :], n_reference, axis=1), x], axis=2
        )
        probabilities = predict_class_probabilities(classifier_packs, points)
        masses.append(
            np.mean(class_probability_ratio(probabilities, 0, 2), axis=1)
        )
    return np.concatenate(masses)


def three_log_zl(classifier_packs, phi_values, n_reference, seed):
    return np.log(three_zl(classifier_packs, phi_values, n_reference, seed))


def highest_density_levels(density, x_grid, y_grid, masses=(0.5, 0.9)):
    density = np.asarray(density, dtype=float)
    cell_mass = density.ravel() * float(np.mean(np.diff(x_grid))) * float(
        np.mean(np.diff(y_grid))
    )
    order = np.argsort(density.ravel())[::-1]
    cumulative = np.cumsum(cell_mass[order])
    levels = []
    for mass in masses:
        index = min(np.searchsorted(cumulative, mass), len(order) - 1)
        levels.append(float(density.ravel()[order[index]]))
    return sorted(levels)


MU_GRID_2D = np.linspace(-3.8, 3.8, 181 if not SMOKE_MODE else 61)
ALPHA_GRID_2D = np.linspace(-3.4, 3.4, 161 if not SMOKE_MODE else 51)
MU_MESH, ALPHA_MESH = np.meshgrid(
    MU_GRID_2D, ALPHA_GRID_2D, indexing="ij"
)
PHI_GRID = np.column_stack([MU_MESH.ravel(), ALPHA_MESH.ravel()])
X_GRID = np.repeat(X_OBS[None, :], len(PHI_GRID), axis=0)
POINTS_GRID = np.column_stack([PHI_GRID, X_GRID]).astype(np.float32)

grid_log_probabilities = predict_class_log_probabilities(three_ce, POINTS_GRID)
log_rp_grid = class_log_ratio(grid_log_probabilities, 0, 1)
log_rl_grid = class_log_ratio(grid_log_probabilities, 0, 2)
log_qp_grid = _flow_log_prob(q_p, PHI_GRID, context=X_GRID)
log_qlm_grid = _flow_log_prob(q_lm, X_GRID, context=PHI_GRID)
log_zp_observed_marginal = float(
    three_log_zp(three_ce, X_OBS, N_NORMALIZATION_CHECK, SEED + 490)[0]
)
posterior_marginal_proposal, _ = normalize_log_surface(
    log_qp_grid.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)
posterior_marginal_hybrid, _ = normalize_log_surface(
    (log_qp_grid + log_rp_grid - log_zp_observed_marginal).reshape(
        MU_MESH.shape
    ),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
posterior_likelihood_proposal_m, _ = normalize_log_surface(
    (design_marginal_logpdf(PHI_GRID) + log_qlm_grid).reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
posterior_likelihood_raw_m, _ = normalize_log_surface(
    (
        design_marginal_logpdf(PHI_GRID) + log_qlm_grid + log_rl_grid
    ).reshape(MU_MESH.shape),
    MU_GRID_2D,
    ALPHA_GRID_2D,
)
log_marginal_truth = design_marginal_logpdf(PHI_GRID) + marginal_log_likelihood(
    X_OBS, PHI_GRID
)
posterior_truth_marginal, _ = normalize_log_surface(
    log_marginal_truth.reshape(MU_MESH.shape), MU_GRID_2D, ALPHA_GRID_2D
)

# A wide deterministic 2D integral gives the full-model evidence because beta
# has already been integrated analytically in p_m(x|mu,alpha).
MU_EVIDENCE_GRID = np.linspace(-10.0, 10.0, 501)
ALPHA_EVIDENCE_GRID = np.linspace(-8.0, 8.0, 401)
MU_EVIDENCE_MESH, ALPHA_EVIDENCE_MESH = np.meshgrid(
    MU_EVIDENCE_GRID, ALPHA_EVIDENCE_GRID, indexing="ij"
)
PHI_EVIDENCE_GRID = np.column_stack([
    MU_EVIDENCE_MESH.ravel(), ALPHA_EVIDENCE_MESH.ravel()
])
_, LOG_EVIDENCE_TRUTH = normalize_log_surface(
    (
        design_marginal_logpdf(PHI_EVIDENCE_GRID)
        + marginal_log_likelihood(X_OBS, PHI_EVIDENCE_GRID)
    ).reshape(MU_EVIDENCE_MESH.shape),
    MU_EVIDENCE_GRID,
    ALPHA_EVIDENCE_GRID,
)


def marginal_surface_marginals(surface):
    return (
        np.trapezoid(surface, ALPHA_GRID_2D, axis=1),
        np.trapezoid(surface, MU_GRID_2D, axis=0),
    )


truth_mu_m, truth_alpha_m = marginal_surface_marginals(
    posterior_truth_marginal
)
proposal_mu_m, proposal_alpha_m = marginal_surface_marginals(
    posterior_marginal_proposal
)
hybrid_mu_m, hybrid_alpha_m = marginal_surface_marginals(
    posterior_marginal_hybrid
)
likelihood_raw_mu_m, likelihood_raw_alpha_m = marginal_surface_marginals(
    posterior_likelihood_raw_m
)
rng = np.random.default_rng(SEED + 510)
n_context_check = 24 if SMOKE_MODE else (80 if FAST_MODE else 160)
theta_check = sample_design(n_context_check, rng)
x_check = simulate(theta_check, rng)
phi_check = sample_design(n_context_check, rng)[:, :MARGINAL_DIM]
heldout_norm = {
    "log_zp": three_log_zp(
        three_ce, x_check, N_NORMALIZATION_CHECK, SEED + 520
    ),
    "log_zl": three_log_zl(
        three_ce, phi_check, N_NORMALIZATION_CHECK, SEED + 521
    ),
}

phi_tail = _draw_conditional(
    q_p,
    X_OBS[None, :],
    2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000),
    SEED + 530,
)[0]
x_tail = np.repeat(X_OBS[None, :], len(phi_tail), axis=0)
tail_points = np.column_stack([phi_tail, x_tail])
tail_probabilities = predict_class_probabilities(three_ce, tail_points)
tail_log_probabilities = np.log(
    np.maximum(tail_probabilities, np.finfo(np.float64).tiny)
)
tail_ratios = class_probability_ratio(tail_probabilities, 0, 1)
tail_summary = importance_tail_summary(
    class_log_ratio(tail_log_probabilities, 0, 1)
)

phi_mode_m = PHI_GRID[np.argmax(posterior_truth_marginal)]
x_likelihood_tail = _draw_conditional(
    q_lm, phi_mode_m[None, :], len(phi_tail), SEED + 531
)[0]
likelihood_tail_points = np.column_stack([
    np.repeat(phi_mode_m[None, :], len(x_likelihood_tail), axis=0),
    x_likelihood_tail,
])
likelihood_tail_probabilities = predict_class_probabilities(
    three_ce, likelihood_tail_points
)
likelihood_tail_log_probabilities = np.log(
    np.maximum(likelihood_tail_probabilities, np.finfo(np.float64).tiny)
)
likelihood_tail_summary = importance_tail_summary(
    class_log_ratio(likelihood_tail_log_probabilities, 0, 2)
)

bridge_three_audit = [
    build_three_bridge_bundle(
        N_BRIDGE_AUDIT_GROUPS,
        N_BRIDGE_AUDIT_INNER,
        N_BRIDGE_AUDIT_MASS_INNER,
        q_p,
        q_lm,
        SEED + 120_000 + 100 * bank,
    )
    for bank in range(N_BRIDGE_AUDIT_BANKS)
]


def heldout_three_bridge_rms(classifier_packs, banks):
    values = []
    for bundle in banks:
        log_probabilities = predict_class_log_probabilities(
            classifier_packs, bundle["bridge_points"]
        )
        zl_probabilities = predict_class_probabilities(
            classifier_packs, bundle["bridge_zl_points"]
        )
        log_zl = np.log(
            np.mean(class_probability_ratio(zl_probabilities, 0, 2), axis=2)
        )
        implied = (
            bundle["bridge_base"]
            + log_probabilities[..., 1]
            - log_probabilities[..., 2]
            - log_zl
        )
        values.append(np.std(implied, axis=1, ddof=1))
    return np.concatenate(values)


heldout_bridge_rms = heldout_three_bridge_rms(three_ce, bridge_three_audit)
del bridge_three_audit

all_log_z = np.concatenate([heldout_norm["log_zp"], heldout_norm["log_zl"]])
marginal_summary = pd.DataFrame([
    {
        "route": "raw marginal q_P",
        "surface IAE": surface_iae(
            posterior_truth_marginal,
            posterior_marginal_proposal,
            MU_GRID_2D,
            ALPHA_GRID_2D,
        ),
    },
    {
        "route": "CE-corrected marginal q_P",
        "surface IAE": surface_iae(
            posterior_truth_marginal,
            posterior_marginal_hybrid,
            MU_GRID_2D,
            ALPHA_GRID_2D,
        ),
    },
    {
        "route": "raw rho q_L^m",
        "surface IAE": surface_iae(
            posterior_truth_marginal,
            posterior_likelihood_proposal_m,
            MU_GRID_2D,
            ALPHA_GRID_2D,
        ),
    },
    {
        "route": "CE-corrected raw rho q_L^m (before Z_L)",
        "surface IAE": surface_iae(
            posterior_truth_marginal,
            posterior_likelihood_raw_m,
            MU_GRID_2D,
            ALPHA_GRID_2D,
        ),
    },
])
display(marginal_summary.style.format(precision=4))
display(pd.DataFrame([{
    "held-out bridge median RMS": float(np.median(heldout_bridge_rms)),
    "held-out bridge q95 RMS": float(np.quantile(heldout_bridge_rms, 0.95)),
    "RMS log Z": float(np.sqrt(np.mean(all_log_z**2))),
    "q95 |log Z|": float(np.quantile(np.abs(all_log_z), 0.95)),
    "P-class probability floor": probability_floor_fraction(tail_probabilities, 1),
    "L-class probability floor": probability_floor_fraction(
        likelihood_tail_probabilities, 2
    ),
    "posterior ESS fraction": tail_summary["ESS_fraction"],
    "posterior Pareto k": tail_summary["pareto_k"],
    "likelihood ESS fraction": likelihood_tail_summary["ESS_fraction"],
    "likelihood Pareto k": likelihood_tail_summary["pareto_k"],
}]).style.format(precision=4))


In [ ]:
DIRECT_COLOR = "#D55E00"
NORM_COLOR = "#0072B2"
POSTHOC_COLOR = "#009E73"
JOINT_CE_COLOR = "#6A3D9A"

truth_levels_m = highest_density_levels(
    posterior_truth_marginal, MU_GRID_2D, ALPHA_GRID_2D
)
qp_levels_m = highest_density_levels(
    posterior_marginal_hybrid, MU_GRID_2D, ALPHA_GRID_2D
)
ql_levels_m = highest_density_levels(
    posterior_likelihood_raw_m, MU_GRID_2D, ALPHA_GRID_2D
)

fig, axes = plt.subplots(2, 3, figsize=(15.0, 8.8), constrained_layout=True)
for axis, learned, levels, title, color in (
    (axes[0, 0], posterior_marginal_hybrid, qp_levels_m,
     "(a) Marginal posterior-flow route", DIRECT_COLOR),
    (axes[0, 1], posterior_likelihood_raw_m, ql_levels_m,
     r"(b) Marginal likelihood-flow route (before $Z_L$)", NORM_COLOR),
):
    axis.contour(
        MU_GRID_2D, ALPHA_GRID_2D, posterior_truth_marginal.T,
        levels=truth_levels_m, colors="black", linewidths=[2.0, 1.4]
    )
    axis.contour(
        MU_GRID_2D, ALPHA_GRID_2D, learned.T,
        levels=levels, colors=color, linewidths=[2.0, 1.4], linestyles="--"
    )
    axis.plot([], [], color="black", lw=1.8, label="analytic truth")
    axis.plot([], [], color=color, lw=1.8, ls="--", label="CE corrected")
    axis.set(xlabel=r"$\mu$", ylabel=r"$\alpha$", title=title)
    axis.legend(fontsize=8)

axes[0, 2].plot(MU_GRID_2D, truth_mu_m, color="black", lw=2.2, label="truth")
axes[0, 2].plot(
    MU_GRID_2D, proposal_mu_m, color="0.55", ls=":", label=r"raw $q_P^m$"
)
axes[0, 2].plot(
    MU_GRID_2D, hybrid_mu_m, color=DIRECT_COLOR, lw=1.9,
    label="CE posterior route"
)
axes[0, 2].plot(
    MU_GRID_2D, likelihood_raw_mu_m, color=NORM_COLOR, ls=":",
    label=r"CE $q_L^m$ (before $Z_L$)"
)
axes[0, 2].set(
    xlabel=r"$\mu$", ylabel="posterior density", title="(c) POI marginal"
)
axes[0, 2].legend(fontsize=7.5)

axes[1, 0].plot(
    ALPHA_GRID_2D, truth_alpha_m, color="black", lw=2.2, label="truth"
)
axes[1, 0].plot(
    ALPHA_GRID_2D, proposal_alpha_m, color="0.55", ls=":", label=r"raw $q_P^m$"
)
axes[1, 0].plot(
    ALPHA_GRID_2D, hybrid_alpha_m, color=DIRECT_COLOR, lw=1.9,
    label="CE posterior route"
)
axes[1, 0].plot(
    ALPHA_GRID_2D, likelihood_raw_alpha_m, color=NORM_COLOR, ls=":",
    label=r"CE $q_L^m$ (before $Z_L$)"
)
axes[1, 0].set(
    xlabel=r"$\alpha$", ylabel="posterior density",
    title="(d) Retained-nuisance marginal"
)
axes[1, 0].legend(fontsize=7.5)

phi_bridge_path = np.column_stack([
    MU_GRID_2D, np.full(len(MU_GRID_2D), phi_mode_m[1])
]).astype(np.float32)
x_bridge_path = np.repeat(
    X_OBS[None, :], len(phi_bridge_path), axis=0
).astype(np.float32)
bridge_path_log_probabilities = predict_class_log_probabilities(
    three_ce, np.column_stack([phi_bridge_path, x_bridge_path])
)
bridge_path_raw = (
    design_marginal_logpdf(phi_bridge_path)
    + _flow_log_prob(q_lm, x_bridge_path, context=phi_bridge_path)
    - _flow_log_prob(q_p, phi_bridge_path, context=x_bridge_path)
    + bridge_path_log_probabilities[:, 1]
    - bridge_path_log_probabilities[:, 2]
)
bridge_path_normalized = (
    bridge_path_raw
    + log_zp_observed_marginal
    - three_log_zl(
        three_ce, phi_bridge_path, N_NORMALIZATION_PATH, SEED + 500
    )
)
axes[1, 1].plot(
    MU_GRID_2D, bridge_path_raw - LOG_EVIDENCE_TRUTH,
    color="0.55", ls=":", label="raw bridge"
)
axes[1, 1].plot(
    MU_GRID_2D,
    bridge_path_normalized - LOG_EVIDENCE_TRUTH,
    color=DIRECT_COLOR,
    lw=1.9,
    label=rf"normalized ({N_NORMALIZATION_PATH:,} draws/point)",
)
axes[1, 1].axhline(0.0, color="black", lw=1)
axes[1, 1].set(
    xlabel=r"$\mu$ at fixed $\alpha_{\rm mode}$",
    ylabel="log-evidence residual",
    title="(e) High-statistics normalization check",
)
axes[1, 1].legend(fontsize=7.5)

rng_plot = np.random.default_rng(SEED + 1)
for index, values in enumerate((heldout_norm["log_zp"], heldout_norm["log_zl"])):
    axes[1, 2].scatter(
        index + rng_plot.normal(0, 0.012, len(values)), values,
        s=12, alpha=0.42, color=DIRECT_COLOR
    )
axes[1, 2].axhline(0.0, color="black", lw=1)
axes[1, 2].set(
    xticks=[0, 1], xticklabels=[r"$\log Z_P^m(x)$", r"$\log Z_L^m(\phi)$"],
    ylabel="held-out conditional log normalizer",
    title="(f) Independent mass closure",
)
for ax in axes.flat:
    ax.grid(alpha=0.22)
export_exercise9_multiclass_figure(fig, "marginal_three_class_ce_posthoc_consistency")
plt.show()


**Figure 1 interpretation.**  Part I is now a true multivariate-flow test: the black contours are the exact posterior after integrating $\beta$, while both learned routes use the same frozen CE classifier.  The likelihood-flow closure is shown before a pointwise $Z_L^m$ correction so that Monte Carlo normalizer noise is not mistaken for learned density structure.  The bridge path and held-out masses use the explicitly reported high-statistics budget and diagnose consistency without affecting training.


## Held-out calibration with one nuisance marginalized

Simulation-based calibration is evaluated separately for the retained coordinates $\mu$ and $\alpha$.  For each held-out simulator draw, samples from $q_P^m(\mu,\alpha\mid x)$ are used both raw and with the frozen CE ratio weights.


In [ ]:
rng = np.random.default_rng(SEED + 1300)
theta_calibration = sample_design(N_CALIBRATION_CONTEXTS, rng)
x_calibration = simulate(theta_calibration, rng)
phi_calibration = _draw_conditional(
    q_p, x_calibration, N_CALIBRATION_SAMPLES, SEED + 1301
)
calibration_points = np.concatenate(
    [
        phi_calibration,
        np.repeat(x_calibration[:, None, :], N_CALIBRATION_SAMPLES, axis=1),
    ],
    axis=2,
)
probabilities = predict_class_probabilities(three_ce, calibration_points)
ratios = class_probability_ratio(probabilities, 0, 1)
weights = normalized_probability_ratios(ratios, axis=1)

part1_pits = {"raw marginal q_P": {}, THREE_CE_LABEL: {}}
for parameter_index, parameter_name in enumerate(("mu", "alpha")):
    below = (
        phi_calibration[..., parameter_index]
        <= theta_calibration[:, parameter_index, None]
    )
    part1_pits["raw marginal q_P"][parameter_name] = below.mean(axis=1)
    part1_pits[THREE_CE_LABEL][parameter_name] = np.sum(weights * below, axis=1)

NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
part1_coverage = {}
calibration_rows = []
for method, parameter_pits in part1_pits.items():
    part1_coverage[method] = {}
    for parameter_name, pit in parameter_pits.items():
        coverage = np.array([
            np.mean(
                (pit >= 0.5 * (1.0 - level))
                & (pit <= 0.5 * (1.0 + level))
            )
            for level in NOMINAL_COVERAGE
        ])
        part1_coverage[method][parameter_name] = coverage
        ordered = np.sort(pit)
        uniform = (np.arange(len(ordered)) + 0.5) / len(ordered)
        calibration_rows.append({
            "method": method,
            "parameter": parameter_name,
            "PIT KS distance": float(np.max(np.abs(ordered - uniform))),
            "max coverage error": float(
                np.max(np.abs(coverage - NOMINAL_COVERAGE))
            ),
        })
display(pd.DataFrame(calibration_rows).style.format(precision=4))

fig, axes = plt.subplots(2, 2, figsize=(11.0, 8.4), constrained_layout=True)
part1_colors = {"raw marginal q_P": "0.55", THREE_CE_LABEL: DIRECT_COLOR}
for column, parameter_name in enumerate(("mu", "alpha")):
    for method in part1_pits:
        pit = part1_pits[method][parameter_name]
        ordered = np.sort(pit)
        empirical = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
        axes[0, column].plot(
            ordered, empirical, color=part1_colors[method], lw=1.8, label=method
        )
        axes[1, column].plot(
            NOMINAL_COVERAGE, part1_coverage[method][parameter_name],
            color=part1_colors[method], lw=1.8, label=method
        )
    axes[0, column].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1, column].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0, column].set(
        xlabel=f"{parameter_name} PIT", ylabel="empirical CDF",
        title=f"({chr(97 + column)}) {parameter_name} PIT"
    )
    axes[1, column].set(
        xlabel="nominal coverage", ylabel="empirical coverage",
        title=f"({chr(99 + column)}) {parameter_name} coverage"
    )
for ax in axes.flat:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.22)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "part1_marginal_2d_heldout_calibration")
plt.show()


# Part II — both nuisances explicit with the original joint dual flows

Part II trains exactly one joint posterior flow and one likelihood flow,

$$
q_P(\mu,\alpha,\beta\mid x),\qquad
q_L(x\mid\mu,\alpha,\beta),
$$

using the same coupling architecture, maximum-likelihood objective, data budgets, and optimizer configuration as Part I and `Exercise_9_multiclass.ipynb`.  The three-class CE model now receives the seven sampled coordinates $(\mu,\alpha,\beta,x_1,\ldots,x_4)$.  No analytic feature or density is added.


In [ ]:
q_lm["flow"].to(torch.device("cpu"))
q_p["flow"].to(torch.device("cpu"))
if torch.cuda.is_available():
    torch.cuda.empty_cache()

rng = np.random.default_rng(SEED + 600)
theta_qp_joint = sample_design(N_FLOW, rng)
x_qp_joint = simulate(theta_qp_joint, rng)
set_torch_seed(SEED + 601)
q_p_joint = train_spline_flow(
    theta_qp_joint,
    context=x_qp_joint,
    checkpoint=MODEL_DIR / "q_p_joint_mu_alpha_beta_given_x.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 601,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_qp_joint, x_qp_joint

rng = np.random.default_rng(SEED + 610)
theta_ql = sample_design(N_FLOW, rng)
x_ql = simulate(theta_ql, rng)
set_torch_seed(SEED + 611)
q_l = train_spline_flow(
    x_ql,
    context=theta_ql,
    checkpoint=MODEL_DIR / "q_l_x_given_mu_alpha_beta.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 611,
    load_if_available=LOAD_IF_AVAILABLE,
    verify_checkpoint_data=True,
)
del theta_ql, x_ql

rng = np.random.default_rng(SEED + 612)
ql_context_pool = sample_design(50_000 if not SMOKE_MODE else 2_000, rng)
ql_audit_contexts, ql_audit_scores, ql_audit_probabilities = (
    select_empirical_context_slices(ql_context_pool)
)
n_audit_contexts = len(ql_audit_contexts)
ql_truth_draws = simulate(
    np.repeat(ql_audit_contexts, N_FLOW_AUDIT_SAMPLES, axis=0), rng
).reshape(n_audit_contexts, N_FLOW_AUDIT_SAMPLES, X_DIM)
ql_tail_audit = sample_only_flow_audit(
    q_l,
    ql_audit_contexts,
    ql_truth_draws,
    ql_audit_scores,
    ql_audit_probabilities,
    r"$q_L(x\mid\mu,\alpha,\beta)$",
    SEED + 613,
)
display(ql_tail_audit.style.format(precision=4))

audit_vector_flow(
    q_p_joint,
    simulate(ql_audit_contexts, rng),
    THETA_DIM,
    r"$q_P(\mu,\alpha,\beta\mid x)$",
    SEED + 614,
)
audit_vector_flow(
    q_l,
    ql_audit_contexts,
    X_DIM,
    r"$q_L(x\mid\mu,\alpha,\beta)$",
    SEED + 615,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def build_joint_three_class_groups(n_groups, q_p_joint, q_l, seed):
    rng = np.random.default_rng(seed)
    theta_s = sample_design(n_groups, rng)
    x_s = simulate(theta_s, rng)
    theta_p = _draw_conditional(q_p_joint, x_s, 1, seed + 1)[:, 0, :]
    x_l = _draw_conditional(q_l, theta_s, 1, seed + 2)[:, 0, :]
    points_s = np.column_stack([theta_s, x_s])
    points_p = np.column_stack([theta_p, x_s])
    points_l = np.column_stack([theta_s, x_l])
    groups = np.stack([points_s, points_p, points_l], axis=1).astype(np.float32)
    return _assert_finite("joint three-class groups", groups, ndim=3)


joint_retry_before = _rqs_retry_count()
joint_three_class_groups = build_joint_three_class_groups(
    N_CLASS, q_p_joint, q_l, SEED + 620
)
print("Joint three-class grouped tensor:", joint_three_class_groups.shape)
print(
    "Inverse-RQS kernel calls retried during class construction:",
    _rqs_retry_count() - joint_retry_before,
)


## Pure CE for the full $(\mu,\alpha,\beta,x)$ classifier

As in Part I, each of ten identical plain MLPs is trained only with equal-prior three-class cross entropy.  There is no dropout, weight decay, layer normalization, auxiliary loss, normalization loss, or bridge loss.


In [ ]:
joint_ce = train_multiclass_classifier(
    joint_three_class_groups,
    n_classes=3,
    checkpoint_dir=MODEL_DIR / "part2_joint_three_class_ce",
    seed_base=SEED + 900,
)
del joint_three_class_groups
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Three-dimensional posterior and likelihood closure

A dense three-dimensional grid would make a high-statistics post-training $Z_L(\theta)$ audit unnecessarily expensive.  Instead, the full closure uses 500,000 independent samples and reports all three one-dimensional marginals plus the three pairwise posterior projections.  Truth samples are prior draws weighted by the exact likelihood; the posterior-flow route uses $q_P$ draws weighted by $r_P$; and the likelihood-flow route uses prior draws weighted by the smooth CE-corrected $q_Lr_L$.  We do **not** divide this dense closure by thousands of independent low-effective-sample estimates of $Z_L(\theta)$.  High-statistics normalization is tested separately on sparse paths and held-out points.  This is a validation construction only.


In [ ]:
def joint_zp(classifier_packs, x_values, n_reference, seed):
    x_values = np.atleast_2d(np.asarray(x_values, dtype=np.float32))
    theta = _draw_conditional(q_p_joint, x_values, n_reference, seed)
    points = np.concatenate(
        [theta, np.repeat(x_values[:, None, :], n_reference, axis=1)], axis=2
    )
    probabilities = predict_class_probabilities(classifier_packs, points)
    return np.mean(class_probability_ratio(probabilities, 0, 1), axis=1)


def joint_log_zp(classifier_packs, x_values, n_reference, seed):
    return np.log(joint_zp(classifier_packs, x_values, n_reference, seed))


def joint_zl(classifier_packs, theta_values, n_reference, seed):
    theta_values = np.atleast_2d(np.asarray(theta_values, dtype=np.float32))
    context_batch = max(1, 65_536 // int(n_reference))
    masses = []
    for start in range(0, len(theta_values), context_batch):
        theta_chunk = theta_values[start : start + context_batch]
        x = _draw_conditional(q_l, theta_chunk, n_reference, seed + start)
        points = np.concatenate(
            [np.repeat(theta_chunk[:, None, :], n_reference, axis=1), x],
            axis=2,
        )
        probabilities = predict_class_probabilities(classifier_packs, points)
        masses.append(
            np.mean(class_probability_ratio(probabilities, 0, 2), axis=1)
        )
    return np.concatenate(masses)


def joint_log_zl(classifier_packs, theta_values, n_reference, seed):
    return np.log(joint_zl(classifier_packs, theta_values, n_reference, seed))


def weighted_histogram_1d(values, weights, edges):
    density, _ = np.histogram(values, bins=edges, weights=weights, density=True)
    return density


def weighted_histogram_2d(x, y, weights, x_edges, y_edges):
    density, _, _ = np.histogram2d(
        x, y, bins=(x_edges, y_edges), weights=weights, density=True
    )
    return density


PARAMETER_NAMES = ("mu", "alpha", "beta")
PARAMETER_LABELS = (r"$\mu$", r"$\alpha$", r"$\beta$")
PARAMETER_EDGES = (
    np.linspace(-4.0, 4.0, 91 if not SMOKE_MODE else 31),
    np.linspace(-3.6, 3.6, 85 if not SMOKE_MODE else 29),
    np.linspace(-3.2, 3.2, 81 if not SMOKE_MODE else 27),
)
PARAMETER_CENTERS = tuple(
    0.5 * (edges[1:] + edges[:-1]) for edges in PARAMETER_EDGES
)

rng = np.random.default_rng(SEED + 980)
theta_reference = sample_design(N_POSTERIOR_REFERENCE, rng)
x_reference = np.repeat(X_OBS[None, :], len(theta_reference), axis=0).astype(
    np.float32
)
truth_weights_joint = normalized_log_weights(
    log_likelihood(X_OBS, theta_reference)
)

theta_joint_draws = _draw_conditional(
    q_p_joint, X_OBS[None, :], N_POSTERIOR_REFERENCE, SEED + 981
)[0]
x_joint_draws = np.repeat(
    X_OBS[None, :], len(theta_joint_draws), axis=0
).astype(np.float32)
posterior_points = np.column_stack([theta_joint_draws, x_joint_draws])
posterior_probabilities = predict_class_probabilities(joint_ce, posterior_points)
posterior_ratios = class_probability_ratio(posterior_probabilities, 0, 1)
proposal_weights_joint = np.full(
    len(theta_joint_draws), 1.0 / len(theta_joint_draws)
)
hybrid_weights_joint = normalized_probability_ratios(posterior_ratios, axis=0)

likelihood_points = np.column_stack([theta_reference, x_reference])
likelihood_log_probabilities = predict_class_log_probabilities(
    joint_ce, likelihood_points
)
log_rl_reference = class_log_ratio(likelihood_log_probabilities, 0, 2)
log_ql_reference = _flow_log_prob(q_l, x_reference, context=theta_reference)
likelihood_proposal_weights_joint = normalized_log_weights(log_ql_reference)
likelihood_raw_weights_joint = normalized_log_weights(
    log_ql_reference + log_rl_reference
)

joint_marginal_densities = {}
for method, samples, weights in (
    ("truth", theta_reference, truth_weights_joint),
    ("raw joint q_P", theta_joint_draws, proposal_weights_joint),
    (JOINT_CE_LABEL, theta_joint_draws, hybrid_weights_joint),
    ("raw rho q_L", theta_reference, likelihood_proposal_weights_joint),
    ("CE rho q_L before Z_L", theta_reference, likelihood_raw_weights_joint),
):
    joint_marginal_densities[method] = [
        weighted_histogram_1d(samples[:, index], weights, PARAMETER_EDGES[index])
        for index in range(THETA_DIM)
    ]

joint_summary_rows = []
for method in (
    "raw joint q_P", JOINT_CE_LABEL, "raw rho q_L",
    "CE rho q_L before Z_L"
):
    row = {"route": method}
    for index, name in enumerate(PARAMETER_NAMES):
        row[f"{name} marginal IAE"] = integrated_absolute_error(
            joint_marginal_densities["truth"][index],
            joint_marginal_densities[method][index],
            PARAMETER_CENTERS[index],
        )
    row["mean marginal IAE"] = float(
        np.mean([row[f"{name} marginal IAE"] for name in PARAMETER_NAMES])
    )
    joint_summary_rows.append(row)
display(pd.DataFrame(joint_summary_rows).style.format(precision=4))

theta_mode = theta_reference[int(np.argmax(truth_weights_joint))]
log_zp_observed = float(
    joint_log_zp(joint_ce, X_OBS, N_NORMALIZATION_CHECK, SEED + 990)[0]
)
print("Importance-reference posterior mode candidate:", theta_mode)


In [ ]:
PAIR_INDEX = ((0, 1), (0, 2), (1, 2))
pair_surfaces = {}
for method, samples, weights in (
    ("truth", theta_reference, truth_weights_joint),
    (JOINT_CE_LABEL, theta_joint_draws, hybrid_weights_joint),
    ("CE rho q_L before Z_L", theta_reference, likelihood_raw_weights_joint),
):
    pair_surfaces[method] = [
        weighted_histogram_2d(
            samples[:, left], samples[:, right], weights,
            PARAMETER_EDGES[left], PARAMETER_EDGES[right]
        )
        for left, right in PAIR_INDEX
    ]

fig, axes = plt.subplots(2, 3, figsize=(15.0, 8.7), constrained_layout=True)
for column, (left, right) in enumerate(PAIR_INDEX):
    for method, color, style, width in (
        ("truth", "black", "-", 2.0),
        (JOINT_CE_LABEL, JOINT_CE_COLOR, "--", 1.8),
        ("CE rho q_L before Z_L", NORM_COLOR, ":", 1.7),
    ):
        density = pair_surfaces[method][column]
        levels = highest_density_levels(
            density, PARAMETER_CENTERS[left], PARAMETER_CENTERS[right]
        )
        axes[0, column].contour(
            PARAMETER_CENTERS[left], PARAMETER_CENTERS[right], density.T,
            levels=levels, colors=color, linestyles=style,
            linewidths=[width, max(1.0, width - 0.4)]
        )
    axes[0, column].plot([], [], color="black", lw=1.8, label="truth")
    axes[0, column].plot(
        [], [], color=JOINT_CE_COLOR, ls="--", lw=1.7,
        label="CE posterior route"
    )
    axes[0, column].plot(
        [], [], color=NORM_COLOR, ls=":", lw=1.7,
        label=r"CE likelihood route (before $Z_L$)"
    )
    axes[0, column].set(
        xlabel=PARAMETER_LABELS[left], ylabel=PARAMETER_LABELS[right],
        title=f"({chr(97 + column)}) pairwise posterior"
    )
    axes[0, column].legend(fontsize=7.2)

for index, (name, label) in enumerate(zip(PARAMETER_NAMES, PARAMETER_LABELS)):
    axes[1, index].plot(
        PARAMETER_CENTERS[index], joint_marginal_densities["truth"][index],
        color="black", lw=2.2, label="truth"
    )
    axes[1, index].plot(
        PARAMETER_CENTERS[index],
        joint_marginal_densities["raw joint q_P"][index],
        color="0.55", ls=":", label=r"raw $q_P$"
    )
    axes[1, index].plot(
        PARAMETER_CENTERS[index], joint_marginal_densities[JOINT_CE_LABEL][index],
        color=JOINT_CE_COLOR, lw=1.9, label="CE posterior route"
    )
    axes[1, index].plot(
        PARAMETER_CENTERS[index],
        joint_marginal_densities["CE rho q_L before Z_L"][index],
        color=NORM_COLOR, ls=":", label=r"CE $q_L$, before $Z_L$"
    )
    axes[1, index].set(
        xlabel=label, ylabel="posterior density",
        title=f"({chr(100 + index)}) {name} marginal"
    )
    axes[1, index].legend(fontsize=7.2)
for ax in axes.flat:
    ax.grid(alpha=0.22)
export_exercise9_multiclass_figure(fig, "joint_3d_three_class_posterior_closure")
plt.show()


## Held-out calibration of the three-dimensional posterior

The same SBC construction is repeated for all three coordinates.  Marginal PIT is not a complete multivariate diagnostic, but it directly tests whether retaining $\beta$ explicitly creates coordinate-wise bias relative to the marginalized two-dimensional experiment.


In [ ]:
rng = np.random.default_rng(SEED + 1400)
theta_joint_calibration = sample_design(N_JOINT_CALIBRATION_CONTEXTS, rng)
x_joint_calibration = simulate(theta_joint_calibration, rng)
theta_joint_calibration_draws = _draw_conditional(
    q_p_joint,
    x_joint_calibration,
    N_JOINT_CALIBRATION_SAMPLES,
    SEED + 1401,
)
joint_calibration_points = np.concatenate(
    [
        theta_joint_calibration_draws,
        np.repeat(
            x_joint_calibration[:, None, :],
            N_JOINT_CALIBRATION_SAMPLES,
            axis=1,
        ),
    ],
    axis=2,
)
joint_probabilities = predict_class_probabilities(
    joint_ce, joint_calibration_points
)
joint_ratios = class_probability_ratio(joint_probabilities, 0, 1)
joint_weights = normalized_probability_ratios(joint_ratios, axis=1)

joint_pits = {"raw joint q_P": {}, JOINT_CE_LABEL: {}}
for parameter_index, parameter_name in enumerate(PARAMETER_NAMES):
    below = (
        theta_joint_calibration_draws[..., parameter_index]
        <= theta_joint_calibration[:, parameter_index, None]
    )
    joint_pits["raw joint q_P"][parameter_name] = below.mean(axis=1)
    joint_pits[JOINT_CE_LABEL][parameter_name] = np.sum(
        joint_weights * below, axis=1
    )

JOINT_NOMINAL_COVERAGE = np.linspace(0.05, 0.95, 19)
joint_coverage = {}
joint_calibration_rows = []
for method, parameter_pits in joint_pits.items():
    joint_coverage[method] = {}
    for parameter_name, pit in parameter_pits.items():
        coverage = np.array([
            np.mean(
                (pit >= 0.5 * (1.0 - level))
                & (pit <= 0.5 * (1.0 + level))
            )
            for level in JOINT_NOMINAL_COVERAGE
        ])
        joint_coverage[method][parameter_name] = coverage
        ordered = np.sort(pit)
        uniform = (np.arange(len(ordered)) + 0.5) / len(ordered)
        joint_calibration_rows.append({
            "method": method,
            "parameter": parameter_name,
            "PIT KS distance": float(np.max(np.abs(ordered - uniform))),
            "max coverage error": float(
                np.max(np.abs(coverage - JOINT_NOMINAL_COVERAGE))
            ),
        })
display(pd.DataFrame(joint_calibration_rows).style.format(precision=4))

fig, axes = plt.subplots(2, 3, figsize=(15.0, 8.3), constrained_layout=True)
joint_colors = {"raw joint q_P": "0.55", JOINT_CE_LABEL: JOINT_CE_COLOR}
for column, parameter_name in enumerate(PARAMETER_NAMES):
    for method in joint_pits:
        pit = joint_pits[method][parameter_name]
        ordered = np.sort(pit)
        empirical = np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
        axes[0, column].plot(
            ordered, empirical, color=joint_colors[method], lw=1.8, label=method
        )
        axes[1, column].plot(
            JOINT_NOMINAL_COVERAGE,
            joint_coverage[method][parameter_name],
            color=joint_colors[method], lw=1.8, label=method
        )
    axes[0, column].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1, column].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0, column].set(
        xlabel=f"{parameter_name} PIT", ylabel="empirical CDF",
        title=f"({chr(97 + column)}) {parameter_name} PIT"
    )
    axes[1, column].set(
        xlabel="nominal coverage", ylabel="empirical coverage",
        title=f"({chr(100 + column)}) {parameter_name} coverage"
    )
for ax in axes.flat:
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.22)
    ax.legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "joint_3d_three_class_calibration")
plt.show()


## Post-training normalization and bridge diagnostics

The full three-parameter conditional masses are estimated only after the CE ensemble is frozen.  Because the likelihood correction has low effective sample size, these checks now use thousands rather than tens of proposal draws per parameter point.  The bridge is inspected along each coordinate through the posterior mode candidate and over independent tail-enriched banks.  Neither $Z_L$ nor the bridge is divided into the primary dense posterior closure: they remain high-statistics consistency checks, not training targets.


In [ ]:
def build_joint_bridge_bundle(
    n_groups, n_inner, n_mass_inner, q_p_joint, q_l, seed
):
    rng = np.random.default_rng(seed)
    theta_anchor = sample_tail_enriched_design(n_groups, rng)
    x_anchor = simulate(theta_anchor, rng)
    theta = _draw_conditional(q_p_joint, x_anchor, n_inner, seed + 1)
    x_repeat = np.repeat(x_anchor[:, None, :], n_inner, axis=1)
    points = np.concatenate([theta, x_repeat], axis=2).astype(np.float32)
    flat_theta = theta.reshape(-1, THETA_DIM)
    flat_x = x_repeat.reshape(-1, X_DIM)
    bridge_base = (
        design_logpdf(flat_theta)
        + _flow_log_prob(q_l, flat_x, context=flat_theta)
        - _flow_log_prob(q_p_joint, flat_theta, context=flat_x)
    ).reshape(n_groups, n_inner)
    x_zl = _draw_conditional(q_l, flat_theta, n_mass_inner, seed + 2)
    theta_zl = np.repeat(flat_theta[:, None, :], n_mass_inner, axis=1)
    zl_points = np.concatenate([theta_zl, x_zl], axis=2).reshape(
        n_groups, n_inner, n_mass_inner, THETA_DIM + X_DIM
    )
    return {"points": points, "base": bridge_base, "zl_points": zl_points}


joint_bridge_banks = [
    build_joint_bridge_bundle(
        N_BRIDGE_AUDIT_GROUPS,
        N_BRIDGE_AUDIT_INNER,
        N_BRIDGE_AUDIT_MASS_INNER,
        q_p_joint,
        q_l,
        SEED + 220_000 + 100 * bank,
    )
    for bank in range(N_BRIDGE_AUDIT_BANKS)
]
joint_bridge_rms = []
for bundle in joint_bridge_banks:
    log_probabilities = predict_class_log_probabilities(joint_ce, bundle["points"])
    zl_probabilities = predict_class_probabilities(
        joint_ce, bundle["zl_points"]
    )
    log_zl = np.log(
        np.mean(class_probability_ratio(zl_probabilities, 0, 2), axis=2)
    )
    implied = (
        bundle["base"]
        + log_probabilities[..., 1]
        - log_probabilities[..., 2]
        - log_zl
    )
    joint_bridge_rms.append(np.std(implied, axis=1, ddof=1))
joint_bridge_rms = np.concatenate(joint_bridge_rms)
del joint_bridge_banks

rng = np.random.default_rng(SEED + 1190)
theta_z_check = sample_design(
    32 if SMOKE_MODE else (80 if FAST_MODE else 160), rng
)
x_z_check = simulate(theta_z_check, rng)
joint_heldout_log_z = {
    r"$\log Z_P(x)$": joint_log_zp(
        joint_ce, x_z_check, N_NORMALIZATION_CHECK, SEED + 1191
    ),
    r"$\log Z_L(\theta)$": joint_log_zl(
        joint_ce, theta_z_check, N_NORMALIZATION_CHECK, SEED + 1192
    ),
}

theta_p_tail = _draw_conditional(
    q_p_joint,
    X_OBS[None, :],
    2_000 if SMOKE_MODE else (20_000 if FAST_MODE else 80_000),
    SEED + 1193,
)[0]
p_tail_points = np.column_stack([
    theta_p_tail, np.repeat(X_OBS[None, :], len(theta_p_tail), axis=0)
])
p_tail_probabilities = predict_class_probabilities(joint_ce, p_tail_points)
p_tail_ratios = class_probability_ratio(p_tail_probabilities, 0, 1)
p_tail_log_probabilities = np.log(
    np.maximum(p_tail_probabilities, np.finfo(np.float64).tiny)
)
p_tail_summary = importance_tail_summary(
    class_log_ratio(p_tail_log_probabilities, 0, 1)
)

x_l_tail = _draw_conditional(
    q_l, theta_mode[None, :], len(theta_p_tail), SEED + 1194
)[0]
l_tail_points = np.column_stack([
    np.repeat(theta_mode[None, :], len(x_l_tail), axis=0), x_l_tail
])
l_tail_probabilities = predict_class_probabilities(joint_ce, l_tail_points)
l_tail_log_probabilities = np.log(
    np.maximum(l_tail_probabilities, np.finfo(np.float64).tiny)
)
l_tail_summary = importance_tail_summary(
    class_log_ratio(l_tail_log_probabilities, 0, 2)
)

joint_diagnostic_rows = []
for name, values in joint_heldout_log_z.items():
    joint_diagnostic_rows.append({
        "diagnostic": name,
        "median |value|": float(np.median(np.abs(values))),
        "q95 |value|": float(np.quantile(np.abs(values), 0.95)),
        "max |value|": float(np.max(np.abs(values))),
        "ESS fraction": np.nan,
        "Pareto k": np.nan,
        "max weight fraction": np.nan,
        "probability-floor fraction": np.nan,
    })
joint_diagnostic_rows.extend([
    {
        "diagnostic": "normalized bridge RMS",
        "median |value|": float(np.median(joint_bridge_rms)),
        "q95 |value|": float(np.quantile(joint_bridge_rms, 0.95)),
        "max |value|": float(np.max(joint_bridge_rms)),
        "ESS fraction": np.nan,
        "Pareto k": np.nan,
        "max weight fraction": np.nan,
        "probability-floor fraction": np.nan,
    },
    {
        "diagnostic": "posterior ratio tail",
        "median |value|": np.nan,
        "q95 |value|": np.nan,
        "max |value|": np.nan,
        "ESS fraction": p_tail_summary["ESS_fraction"],
        "Pareto k": p_tail_summary["pareto_k"],
        "max weight fraction": p_tail_summary["max_weight_fraction"],
        "probability-floor fraction": probability_floor_fraction(
            p_tail_probabilities, 1
        ),
    },
    {
        "diagnostic": "likelihood ratio tail",
        "median |value|": np.nan,
        "q95 |value|": np.nan,
        "max |value|": np.nan,
        "ESS fraction": l_tail_summary["ESS_fraction"],
        "Pareto k": l_tail_summary["pareto_k"],
        "max weight fraction": l_tail_summary["max_weight_fraction"],
        "probability-floor fraction": probability_floor_fraction(
            l_tail_probabilities, 2
        ),
    },
])
display(pd.DataFrame(joint_diagnostic_rows).style.format(precision=4))


In [ ]:
PATH_GRIDS = (
    np.linspace(-3.8, 3.8, 181),
    np.linspace(-3.4, 3.4, 161),
    np.linspace(-3.0, 3.0, 151),
)


def parameter_path(parameter_index, values):
    path = np.repeat(theta_mode[None, :], len(values), axis=0)
    path[:, parameter_index] = values
    return path.astype(np.float32)


def joint_bridge_path(theta_path, seed):
    x = np.repeat(X_OBS[None, :], len(theta_path), axis=0)
    points = np.column_stack([theta_path, x])
    log_probabilities = predict_class_log_probabilities(joint_ce, points)
    raw = (
        design_logpdf(theta_path)
        + _flow_log_prob(q_l, x, context=theta_path)
        - _flow_log_prob(q_p_joint, theta_path, context=x)
        + log_probabilities[:, 1]
        - log_probabilities[:, 2]
    )
    normalized = (
        raw
        + log_zp_observed
        - joint_log_zl(
            joint_ce, theta_path, N_NORMALIZATION_PATH, seed
        )
    )
    return raw, normalized


bridge_paths = [
    joint_bridge_path(
        parameter_path(index, values), SEED + 1200 + index
    )
    for index, values in enumerate(PATH_GRIDS)
]

fig, axes = plt.subplots(2, 3, figsize=(15.0, 8.5), constrained_layout=True)
for index, (values, label, (raw, normalized)) in enumerate(
    zip(PATH_GRIDS, PARAMETER_LABELS, bridge_paths)
):
    axes[0, index].plot(
        values, raw - LOG_EVIDENCE_TRUTH, color="0.55", ls=":",
        label="raw bridge"
    )
    axes[0, index].plot(
        values, normalized - LOG_EVIDENCE_TRUTH,
        color=JOINT_CE_COLOR, lw=1.9,
        label=rf"normalized ({N_NORMALIZATION_PATH:,}/point)"
    )
    axes[0, index].axhline(0, color="black", lw=1)
    axes[0, index].set(
        xlabel=label, ylabel="log-evidence residual",
        title=f"({chr(97 + index)}) bridge along {PARAMETER_NAMES[index]}"
    )
    axes[0, index].legend(fontsize=8)

rng_plot = np.random.default_rng(SEED + 1204)
for index, (label, values) in enumerate(joint_heldout_log_z.items()):
    axes[1, 0].scatter(
        index + rng_plot.normal(0, 0.012, len(values)), values,
        s=12, alpha=0.4, color=JOINT_CE_COLOR
    )
axes[1, 0].axhline(0, color="black", lw=1)
axes[1, 0].set(
    xticks=[0, 1], xticklabels=list(joint_heldout_log_z),
    ylabel="held-out log normalizer",
    title=rf"(d) Mass checks ({N_NORMALIZATION_CHECK:,}/point)"
)

normalized_tail_ratios = p_tail_ratios / np.mean(p_tail_ratios)
ordered = np.sort(np.maximum(normalized_tail_ratios, np.finfo(float).tiny))
survival = 1.0 - np.arange(1, len(ordered) + 1) / (len(ordered) + 1)
axes[1, 1].plot(ordered, survival, color=JOINT_CE_COLOR, lw=1.8)
axes[1, 1].set(
    xscale="log", yscale="log", xlabel=r"normalized $d_S/d_P$",
    ylabel="empirical survival", title="(e) Posterior-correction tail"
)

axes[1, 2].hist(
    joint_bridge_rms, bins=40, density=True, histtype="stepfilled",
    color=JOINT_CE_COLOR, alpha=0.25
)
axes[1, 2].axvline(
    np.median(joint_bridge_rms), color=JOINT_CE_COLOR, lw=1.8,
    label="median"
)
axes[1, 2].set(
    xlabel="within-context bridge RMS", ylabel="density",
    title="(f) Independent bridge banks"
)
axes[1, 2].legend(fontsize=8)
for ax in axes.flat:
    ax.grid(alpha=0.22)
export_exercise9_multiclass_figure(
    fig, "joint_3d_three_class_consistency_and_normalization"
)
plt.show()


## Systematic-prior and auxiliary-measurement update

Because both nuisances are explicit, a frozen corrected posterior can be updated in both $\alpha$ and $\beta$ without simulator calls or retraining.  We replace their design priors by narrower Gaussian priors and add independent auxiliary measurements.  The learned and exact updates use the same importance samples but independent learned versus analytic weights.


In [ ]:
ALPHA_PRIOR_MEAN, ALPHA_PRIOR_SIGMA = 0.30, 0.45
BETA_PRIOR_MEAN, BETA_PRIOR_SIGMA = -0.20, 0.40
A_OBSERVED, SIGMA_A = 0.10, 0.25
B_OBSERVED, SIGMA_B = -0.05, 0.22


def systematic_log_update(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    alpha, beta = theta[:, 1], theta[:, 2]
    return (
        norm.logpdf(alpha, ALPHA_PRIOR_MEAN, ALPHA_PRIOR_SIGMA)
        - design_alpha_logpdf(alpha)
        + norm.logpdf(A_OBSERVED, alpha, SIGMA_A)
        + norm.logpdf(beta, BETA_PRIOR_MEAN, BETA_PRIOR_SIGMA)
        - design_beta_logpdf(beta)
        + norm.logpdf(B_OBSERVED, beta, SIGMA_B)
    )


truth_update_weights = normalized_log_weights(
    np.log(np.maximum(truth_weights_joint, 1.0e-300))
    + systematic_log_update(theta_reference)
)
learned_update_weights = normalized_log_weights(
    np.log(np.maximum(hybrid_weights_joint, 1.0e-300))
    + systematic_log_update(theta_joint_draws)
)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2), constrained_layout=True)
for index, (name, label) in enumerate(zip(PARAMETER_NAMES, PARAMETER_LABELS)):
    truth_update_density = weighted_histogram_1d(
        theta_reference[:, index], truth_update_weights, PARAMETER_EDGES[index]
    )
    learned_update_density = weighted_histogram_1d(
        theta_joint_draws[:, index], learned_update_weights,
        PARAMETER_EDGES[index]
    )
    axes[index].plot(
        PARAMETER_CENTERS[index], truth_update_density,
        color="black", lw=2.2, label="truth"
    )
    axes[index].plot(
        PARAMETER_CENTERS[index], learned_update_density,
        color=JOINT_CE_COLOR, lw=1.9, label="updated joint hNPE"
    )
    axes[index].set(
        xlabel=label, ylabel="posterior density",
        title=f"({chr(97 + index)}) updated {name}"
    )
    axes[index].grid(alpha=0.22)
    axes[index].legend(fontsize=8)
export_exercise9_multiclass_figure(fig, "joint_3d_systematic_update")
plt.show()


# Applications of the frozen three-parameter dual model

As in `Exercise_9_multiclass.ipynb`, the frozen $q_P$, $q_L$, and CE ensemble are reused for corrected generation, absolute evidence, posterior prediction, and selection integrals.  The following cells add no training and make no simulator-density call except where an explicit analytic validation curve is labelled as truth.


In [ ]:
N_APPLICATION = 256 if SMOKE_MODE else (2_048 if FAST_MODE else 4_096)
N_GENERATIVE = 512 if SMOKE_MODE else (5_000 if FAST_MODE else 20_000)


def corrected_likelihood_candidates(theta, n_candidates, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = _draw_conditional(q_l, theta, n_candidates, seed)
    points = np.concatenate(
        [np.repeat(theta[:, None, :], n_candidates, axis=1), x], axis=2
    )
    probabilities = predict_class_probabilities(joint_ce, points)
    ratios = class_probability_ratio(probabilities, 0, 2)
    weights = normalized_probability_ratios(ratios, axis=1)
    return x, weights, np.log(np.mean(ratios, axis=1))


def importance_resample(values, weights, seed):
    values = np.asarray(values)
    weights = np.asarray(weights, dtype=float)
    rng = np.random.default_rng(seed)
    index = rng.choice(
        len(values), size=len(values), replace=True, p=weights / weights.sum()
    )
    return values[index]


def joint_zl_budget_scan(classifier_packs, theta_values, budgets, seed):
    """Estimate every budget from one nested proposal bank per theta."""
    theta_values = np.atleast_2d(
        np.asarray(theta_values, dtype=np.float32)
    )
    budgets = tuple(sorted({int(value) for value in budgets}))
    if not budgets or budgets[0] < 1:
        raise ValueError("Every normalization budget must be positive.")
    maximum = budgets[-1]
    estimates = np.empty(
        (len(budgets), len(theta_values)), dtype=np.float64
    )
    context_batch = max(1, 65_536 // maximum)
    for start in range(0, len(theta_values), context_batch):
        theta_chunk = theta_values[start : start + context_batch]
        x = _draw_conditional(q_l, theta_chunk, maximum, seed + start)
        points = np.concatenate([
            np.repeat(theta_chunk[:, None, :], maximum, axis=1), x
        ], axis=2)
        probabilities = predict_class_probabilities(
            classifier_packs, points
        )
        ratios = class_probability_ratio(probabilities, 0, 2)
        cumulative = np.cumsum(ratios, axis=1, dtype=np.float64)
        estimates[:, start : start + len(theta_chunk)] = np.stack([
            cumulative[:, budget - 1] / float(budget)
            for budget in budgets
        ])
    if not np.isfinite(estimates).all() or np.any(estimates <= 0.0):
        raise FloatingPointError(
            "Invalid conditional mass in the nested budget scan."
        )
    return np.asarray(budgets, dtype=int), estimates


## Corrected generation, evidence, and posterior prediction

Likelihood-flow draws are corrected with $r_L$.  The primary posterior-predictive constructions use the smooth CE corrections before a noisy pointwise $Z_L$ division.  Absolute evidence is shown both before $Z_L$ and through an explicit **inner-normalization budget scan** using nested proposal banks with 512, 2,048, 8,192, and 20,000 draws per parameter point in the full run.  This separates outer hNPE Monte Carlo convergence from the inner conditional-mass convergence that caused the previous bias.  The fourth observable is treated exactly like the first three.


In [ ]:
APP_THETA = np.array([
    theta_mode,
    [-1.35, 0.75, -0.40],
    [2.10, -0.60, 0.80],
], dtype=np.float32)
rng = np.random.default_rng(SEED + 1500)
proposal_generations, corrected_generations, truth_generations = [], [], []
generative_rows = []
for index, theta in enumerate(APP_THETA):
    candidates, weights, log_z = corrected_likelihood_candidates(
        theta[None, :], N_GENERATIVE, SEED + 1501 + index
    )
    corrected = importance_resample(
        candidates[0], weights[0], SEED + 1510 + index
    )
    truth = simulate(np.repeat(theta[None, :], N_GENERATIVE, axis=0), rng)
    proposal_generations.append(candidates[0])
    corrected_generations.append(corrected)
    truth_generations.append(truth)
    generative_rows.append({
        "theta": tuple(theta),
        "log Z_L": float(log_z[0]),
        "proposal mean W1": float(np.mean([
            wasserstein_distance(candidates[0, :, j], truth[:, j])
            for j in range(X_DIM)
        ])),
        "corrected mean W1": float(np.mean([
            wasserstein_distance(corrected[:, j], truth[:, j])
            for j in range(X_DIM)
        ])),
    })
display(pd.DataFrame(generative_rows).style.format(precision=4))

theta_app = _draw_conditional(
    q_p_joint, X_OBS[None, :], N_APPLICATION, SEED + 1530
)[0].astype(np.float32)
x_obs_app = np.repeat(X_OBS[None, :], N_APPLICATION, axis=0).astype(np.float32)
obs_points = np.column_stack([theta_app, x_obs_app])
obs_probabilities = predict_class_probabilities(joint_ce, obs_points)
obs_log_probabilities = np.log(
    np.maximum(obs_probabilities, np.finfo(np.float64).tiny)
)
rp_obs = class_probability_ratio(obs_probabilities, 0, 1)
log_rl_obs = class_log_ratio(obs_log_probabilities, 0, 2)
log_qp_app = _flow_log_prob(q_p_joint, theta_app, context=x_obs_app)
log_ql_app = _flow_log_prob(q_l, x_obs_app, context=theta_app)
log_likelihood_before_z = log_ql_app + log_rl_obs
log_evidence_weight_before_z = (
    design_logpdf(theta_app) + log_likelihood_before_z - log_qp_app
)
learned_log_evidence_before_z = float(
    logsumexp(log_evidence_weight_before_z) - np.log(N_APPLICATION)
)
evidence_theta = theta_app[:N_EVIDENCE_OUTER]
evidence_base_weight = log_evidence_weight_before_z[:N_EVIDENCE_OUTER]
evidence_budgets, evidence_zl = joint_zl_budget_scan(
    joint_ce, evidence_theta, EVIDENCE_Z_BUDGETS, SEED + 11_530
)
evidence_trace = np.array([
    logsumexp(evidence_base_weight - np.log(zl))
    - np.log(N_EVIDENCE_OUTER)
    for zl in evidence_zl
])
highest_stat_evidence_weight = (
    evidence_base_weight - np.log(evidence_zl[-1])
)
evidence_tail = importance_tail_summary(highest_stat_evidence_weight)
print(
    f"absolute log evidence: before Z_L={learned_log_evidence_before_z:.5f}, "
    f"highest-stat Z_L={evidence_trace[-1]:.5f}, "
    f"truth={LOG_EVIDENCE_TRUTH:.5f}; "
    f"inner draws/theta={evidence_budgets[-1]:,}, "
    f"ESS fraction={evidence_tail['ESS_fraction']:.4f}, "
    f"Pareto k={evidence_tail['pareto_k']:.3f}"
)
display(pd.DataFrame({
    "inner q_L draws / theta": evidence_budgets,
    "learned log evidence": evidence_trace,
    "learned minus truth": evidence_trace - LOG_EVIDENCE_TRUTH,
}).style.format(precision=5))

x_rep = _draw_conditional(q_l, theta_app, 1, SEED + 1540)[:, 0, :]
rep_points = np.column_stack([theta_app, x_rep])
rep_probabilities = predict_class_probabilities(joint_ce, rep_points)
rep_log_probabilities = np.log(
    np.maximum(rep_probabilities, np.finfo(np.float64).tiny)
)
rl_rep = class_probability_ratio(rep_probabilities, 0, 2)
log_rl_rep = class_log_ratio(rep_log_probabilities, 0, 2)

predictive_hnpe_weights = normalized_probability_ratios(
    rp_obs * rl_rep, axis=0
)
predictive_hnde_weights = normalized_log_weights(
    log_evidence_weight_before_z + log_rl_rep
)
predictive_hnpe = importance_resample(
    x_rep, predictive_hnpe_weights, SEED + 1541
)
predictive_hnde = importance_resample(
    x_rep, predictive_hnde_weights, SEED + 1542
)
rng_truth = np.random.default_rng(SEED + 1543)
truth_index = rng_truth.choice(
    len(theta_reference), size=N_APPLICATION, replace=True, p=truth_weights_joint
)
predictive_truth = simulate(theta_reference[truth_index], rng_truth)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.3), constrained_layout=True)
for proposal, corrected, truth in zip(
    proposal_generations, corrected_generations, truth_generations
):
    axes[0].hist(truth[:, 1], bins=55, density=True, histtype="step", lw=1.6)
    axes[0].hist(
        proposal[:, 1], bins=55, density=True, histtype="step", lw=1.0, ls=":"
    )
    axes[0].hist(
        corrected[:, 1], bins=55, density=True, histtype="step", lw=1.5, ls="--"
    )
axes[0].set(
    xlabel=r"$x_2$", ylabel="density",
    title="(a) Corrected likelihood generation"
)

axes[1].plot(
    evidence_budgets, evidence_trace, marker="o", color=JOINT_CE_COLOR,
    label=rf"normalized ({N_EVIDENCE_OUTER:,} outer draws)"
)
axes[1].axhline(
    learned_log_evidence_before_z, color=NORM_COLOR, ls=":",
    label=rf"before $Z_L$ ({N_APPLICATION:,} outer draws)"
)
axes[1].axhline(LOG_EVIDENCE_TRUTH, color="black", ls="--", label="truth")
axes[1].set(
    xscale="log", xlabel=r"inner $q_L$ draws per $\theta$",
    ylabel="log evidence", title="(b) Inner-normalization scan"
)
axes[1].legend(fontsize=8)

axes[2].hist(
    predictive_truth[:, 1], bins=50, density=True, histtype="step",
    color="black", lw=2.0, label="truth"
)
axes[2].hist(
    predictive_hnpe[:, 1], bins=50, density=True, histtype="step",
    color=JOINT_CE_COLOR, lw=1.7, label=r"hNPE route (before $Z_L$)"
)
axes[2].hist(
    predictive_hnde[:, 1], bins=50, density=True, histtype="step",
    color=POSTHOC_COLOR, lw=1.7, ls="--",
    label=r"hNDE route (before $Z_L$)"
)
axes[2].set(
    xlabel=r"$x_2^{\rm rep}$", ylabel="density",
    title="(c) Posterior predictive"
)
axes[2].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.22)
export_exercise9_multiclass_figure(
    fig, "joint_3d_generation_evidence_predictive"
)
plt.show()


## Selection integrals without simulator calls

To keep this diagnostic visually comparable with the original exercise, we show a $(\mu,\alpha)$ surface at fixed $\beta=\beta_{\rm mode}$.  The corrected hNDE integrates a linear selection over its own draws.  The full run now uses 8,192 proposal draws per grid point rather than 512, raising the expected effective likelihood-correction count at the posterior mode from about 29 to about 456.  The analytic Gaussian-mixture result appears only as validation truth.


In [ ]:
N_SELECTION_GRID = 13 if SMOKE_MODE else (25 if FAST_MODE else 41)
MU_SELECTION = np.linspace(-3.2, 3.2, N_SELECTION_GRID)
ALPHA_SELECTION = np.linspace(-2.6, 2.6, N_SELECTION_GRID)
MU_SEL_MESH, ALPHA_SEL_MESH = np.meshgrid(
    MU_SELECTION, ALPHA_SELECTION, indexing="ij"
)
BETA_SELECTION = float(theta_mode[2])
THETA_SELECTION = np.column_stack([
    MU_SEL_MESH.ravel(),
    ALPHA_SEL_MESH.ravel(),
    np.full(MU_SEL_MESH.size, BETA_SELECTION),
]).astype(np.float32)


def learned_selection_efficiency(theta, n_draws, seed):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = _draw_conditional(q_l, theta, n_draws, seed)
    points = np.concatenate(
        [np.repeat(theta[:, None, :], n_draws, axis=1), x], axis=2
    )
    probabilities = predict_class_probabilities(joint_ce, points)
    ratios = class_probability_ratio(probabilities, 0, 2)
    selected = (x[..., 1] - 0.6 * x[..., 2] + 0.25 * x[..., 3]) > 0.5
    return np.sum(ratios * selected, axis=1) / np.sum(ratios, axis=1)


beta_learned = learned_selection_efficiency(
    THETA_SELECTION, N_SELECTION_DRAWS, SEED + 201_560
).reshape(MU_SEL_MESH.shape)

base_selection_mean = (
    0.72 * MU_SEL_MESH**2
    - 0.6 * 0.8 * np.cos(MU_SEL_MESH)
    + 0.25 * 0.55 * np.sin(0.8 * MU_SEL_MESH)
    + (-0.4 - 0.6 * 0.3 + 0.25 * 0.5) * ALPHA_SEL_MESH
    + (0.55 - 0.6 * 0.65 + 0.25 * -0.45) * BETA_SELECTION
)
selection_offset = (
    SIMULATOR_OFFSETS[:, 1]
    - 0.6 * SIMULATOR_OFFSETS[:, 2]
    + 0.25 * SIMULATOR_OFFSETS[:, 3]
)
selection_sigma = np.sqrt(
    SIMULATOR_SIGMAS[:, 1] ** 2
    + 0.6**2 * SIMULATOR_SIGMAS[:, 2] ** 2
    + 0.25**2 * SIMULATOR_SIGMAS[:, 3] ** 2
)
beta_truth = sum(
    weight * (
        1.0 - norm.cdf(
            (0.5 - base_selection_mean - offset) / sigma
        )
    )
    for weight, offset, sigma in zip(
        SIMULATOR_WEIGHTS, selection_offset, selection_sigma
    )
)

fig, axes = plt.subplots(1, 3, figsize=(13.7, 4.2), constrained_layout=True)
extent = [
    ALPHA_SELECTION[0], ALPHA_SELECTION[-1],
    MU_SELECTION[0], MU_SELECTION[-1]
]
im0 = axes[0].imshow(
    beta_truth, origin="lower", aspect="auto", extent=extent,
    vmin=0, vmax=1, cmap="viridis"
)
axes[1].imshow(
    beta_learned, origin="lower", aspect="auto", extent=extent,
    vmin=0, vmax=1, cmap="viridis"
)
residual = beta_learned - beta_truth
bound = max(0.02, float(np.max(np.abs(residual))))
im2 = axes[2].imshow(
    residual, origin="lower", aspect="auto", extent=extent,
    vmin=-bound, vmax=bound, cmap="coolwarm"
)
axes[0].set_title("(a) Analytic mixture truth")
axes[1].set_title("(b) Corrected joint hNDE")
axes[2].set_title("(c) Learned minus truth")
for ax in axes:
    ax.set(xlabel=r"$\alpha$", ylabel=r"$\mu$")
fig.suptitle(
    rf"Selection surface at fixed $\beta={BETA_SELECTION:.2f}$ "
    rf"({N_SELECTION_DRAWS:,} proposal draws/point)"
)
fig.colorbar(im0, ax=axes[:2], label=r"selection efficiency $\varepsilon$")
fig.colorbar(im2, ax=axes[2], label="efficiency residual")
export_exercise9_multiclass_figure(fig, "joint_3d_selection_integral")
plt.show()


## Conclusions and run checklist

This companion notebook changes the statistical benchmark, not the hybrid method:

1. Part I marginalizes only $\beta$ and trains the two-dimensional coupling flow $q_P^m(\mu,\alpha\mid x)$ together with $q_L^m(x\mid\mu,\alpha)$;
2. the hidden-$\beta$ validation likelihood is an exact four-component multivariate Gaussian mixture, but no analytic density is used in training;
3. Part II trains the full original dual construction $q_P(\mu,\alpha,\beta\mid x)$ and $q_L(x\mid\mu,\alpha,\beta)$;
4. every classifier is the unchanged ten-member, 1024-wide plain MLP ensemble trained with equal-prior multiclass CE only and the same stepped $10^{-4}\!\to10^{-9}$ schedule;
5. dropout, weight decay, layer normalization, auxiliary losses, normalization losses, bridge losses, and explicit exponentiation of logit differences remain absent;
6. dense posterior and predictive closures use the CE correction before a noisy pointwise $Z_L$ division; conditional normalization and bridge relations are high-statistics post-training checks on sparse points and paths;
7. this committed rerun is checkpoint-only: it verifies all four flow and twenty classifier files before doing any work and stops rather than retraining if one is absent.

For a paper run, inspect both the two-dimensional exact surface closure and the three-dimensional pairwise/marginal closure, probability floors, ESS/Pareto-$k$, conditional masses, bridge variation, and SBC before interpreting downstream applications.
